# NER 第 5a 步：记忆基线

版本：**2026-11-03 memory-v1-envfix1**。修正 Python 兼容性检查与安装日志；模型和评估规则不变。

顺序：**5a.1 → 5a.7 → 下载 ZIP 和 notebook → 5a.8 确认**。使用 **Python 3.12、CPU** 运行时。
不挂载 Drive，不使用 bucket、API key 或 OAuth。数据缺失时恢复已验收的固定版本。

仅从训练集完整实体构建词典，剔除同一短语对应多个实体类型的条目。
预测只接收原始词序列：区分大小写、整词匹配，不读取 gold/POS/chunk。
所有候选按**词数从长到短**接受；重叠时长者优先，等长时先出现者优先；不跨句子。
“O”不是实体类型，同一短语作为非实体出现不等同于两个实体类型冲突。
上述大小写与重叠细节在评分前固定，无参数搜索。

本步先冻结规则与完整词典，再运行开发集和该固定基线的一次最终测试。
使用第四步原封不动的实体级评估、预测 JSONL 与 2,000 次句子 bootstrap（95%，seed 2026）。
输出整体及四类型 P/R/F1、词典统计、置信区间和历史参照表。

[原论文 §3.4、表 5](https://aclanthology.org/W03-0419.pdf)的英文基线为 P=71.91、R=50.90、F1=59.61（百分制）。
它是历史参照，**不是预填的实验结果或必须凑到的验收阈值**。
源数据副本与具体重叠实现的差异需要如实报告；若分数差异较大，保留原结果并反馈，不根据测试分数更改规则。
本实现依据论文文字定义，不声称与历史可执行程序完全相同。

同一运行时重跑自动复用有校验的预测和评分。若已有本步 ZIP 而换了运行时，先用 Files 面板上传 ZIP，
在 5a.2 填写恢复路径；不要为了改变分数反复测试。首次运行保持恢复路径为 None。


## 5a.1 检查依赖
先打印 Python 版本，再核对固定依赖和逐行显示安装日志。请选择 Python 3.12；可复用第四步成功使用的运行时。
Python 3.13 不适用于本项目的 numpy 2.0.2 / pandas 2.2.2 固定环境，检测到后立即停止，不会尝试安装。
仅重启会话不会改变 Python 大版本；需连接已有兼容运行时，或由 Python 3.12 模板创建新运行时。
如在兼容运行时安装成功，按提示重启 Python 会话后从本节重新开始。无需 GPU。

版本依据：[NumPy 2.0.2](https://numpy.org/doc/stable/release/2.0.2-notes.html)、
[pandas 3.13 兼容性](https://pandas.pydata.org/docs/whatsnew/v2.2.3.html)。


In [ ]:
"""Colab cell: guard Python compatibility, show installation progress, retain pins."""
import importlib
import importlib.metadata as metadata
import os
import signal
import subprocess
import sys

REQUIRED = {'datasets':'4.0.0','numpy':'2.0.2','pandas':'2.2.2','seqeval':'1.2.2',
            'scikit-learn':'1.6.1','huggingface-hub':'0.36.2'}
print('Python:', sys.version.split()[0], '| executable:', sys.executable, flush=True)
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError(
        'STOP BEFORE INSTALL: this project pins numpy==2.0.2 and pandas==2.2.2. '
        'Use a Python 3.12 runtime (for example the runtime used successfully for Step 4). '
        'Restarting this Python 3.13+ session will not change its Python version. '
        'Connect to a compatible runtime, then run 5a.1 again. No package installation was started.')

missing = []
for package, version in REQUIRED.items():
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    print(f'{package}: installed={current or "missing"}; required={version}', flush=True)
    if current != version:
        missing.append(package)

if missing:
    print('Installing fixed requirements; mismatched packages:', ', '.join(missing), flush=True)
    print('Live pip output follows. After successful installation, restart Python and rerun 5a.1.', flush=True)
    command = [sys.executable, '-u', '-m', 'pip', 'install', '--use-pep517',
               '--only-binary=numpy,pandas,scipy,scikit-learn,pyarrow',
               '--progress-bar', 'off', '--disable-pip-version-check', '--no-input',
               '--timeout', '30', '--retries', '2',
               *[f'{p}=={v}' for p,v in REQUIRED.items()]]
    # Compiled scientific packages require wheels: fail clearly instead of an
    # unexpected source build. seqeval's small source distribution is allowed.
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=(os.name == 'posix'))
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        # Stop only this installer and its child build processes, not other jobs.
        if process.poll() is None:
            if os.name == 'posix':
                try: os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError: pass
            else:
                process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                if os.name == 'posix':
                    try: os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError: pass
                else:
                    process.kill()
                process.wait()
        print('\nInstallation interrupted. Stop here; send the last pip output before continuing.', flush=True)
        raise
    finally:
        process.stdout.close()
    if return_code:
        raise RuntimeError(f'pip failed (exit {return_code}). Send the last pip lines; do not run 5a.2 yet.')
    raise RuntimeError('Installation finished successfully. Restart the Python session, then rerun 5a.1.')

for package, version in REQUIRED.items():
    module = importlib.import_module({'scikit-learn':'sklearn','huggingface-hub':'huggingface_hub'}.get(package,package))
    imported = getattr(module,'__version__',None)
    if imported is not None and imported != version:
        raise RuntimeError('Stale imported package: restart the Python session, then rerun 5a.1.')
import numpy as np
import pandas as pd
print('Dependencies ready:', REQUIRED, flush=True)


## 5a.2 初始化与公共代码
首次运行保持默认配置。完整重跑从本节开始，每次生成独立结果目录，冻结模型和已保存预测复用。
只有恢复**你自己已下载的 5a ZIP**时才填写 `RESTORE_STEP5A_ZIP`；恢复到新目录，保留已有文件。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import platform
import uuid
import hashlib

NOTEBOOK_REVISION='2026-11-03-memory-v1-envfix1'
PROJECT_ROOT=Path.home()/'cs5489_ner_local'
EXPORT_DIR=Path.home()/'ner_exports'
RESTORE_STEP5A_ZIP=None  # e.g. '/root/NER_step5a_....zip'; first run: leave None.
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
STEP5A_COMPLETE=DATA_READY=MODEL_FROZEN=VALIDATION_READY=TEST_READY=BACKUP_VERIFIED=BACKUP_DOWNLOADED=False


In [ ]:
SOURCE_BUNDLE = {'data.py': '"""Stable shared data entry points; source loading remains in step1_data.py."""\nfrom metrics import LABELS\n\n\ndef label_mapping(dataset):\n    from step1_data import label_names\n    names=label_names(dataset[\'train\'])[\'ner_tags\']\n    if set(names)!=LABELS or any(label_names(split)[\'ner_tags\']!=names for split in dataset.values()):\n        raise ValueError(\'Label metadata differs across splits or from the metric contract.\')\n    return dict(label_names=names,label2id={name:i for i,name in enumerate(names)},\n                id2label={i:name for i,name in enumerate(names)},scheme=\'IOB2\')\n\n\ndef load_verified_dataset(project_root,accepted_reference,run_id):\n    from step2_eda import ensure_dataset\n    dataset,record=ensure_dataset(project_root,accepted_reference,run_id)\n    return dataset,record,label_mapping(dataset)\n\n\ndef decoded_gold(rows,label_names):\n    """Only call on the authorized split; this function does not choose a split."""\n    return [[label_names[v] for v in row[\'ner_tags\']] for row in rows]\n', 'experiment_protocol.py': '"""Contiguous document-safe CV folds and an immutable pre-training contract."""\nimport hashlib\nimport json\nfrom numbers import Integral\nfrom pathlib import Path\nfrom metrics import METRIC_CONTRACT\nfrom prediction_io import SCHEMA\n\nPROTOCOL_VERSION=\'ner-experiment-v1\'\nSEEDS=[42,52,62]\nBOOTSTRAP_SEED=2026\n\n\ndef canonical_bytes(value):\n    return (json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'))+\'\\n\').encode(\'utf-8\')\n\n\ndef fingerprint(value):\n    return hashlib.sha256(canonical_bytes(value)).hexdigest()\n\n\ndef _document_starts(rows):\n    starts=[]; seen=set(); previous=None\n    for i,row in enumerate(rows):\n        doc=row.get(\'document_id\')\n        if isinstance(doc,bool) or not isinstance(doc,Integral):\n            raise ValueError(\'Document-safe folds require integer document_id metadata.\')\n        if i==0 or doc!=previous:\n            if doc in seen:\n                raise ValueError(\'A document appears in noncontiguous sentence runs.\')\n            seen.add(doc); starts.append(i)\n        previous=doc\n    return starts\n\n\ndef make_contiguous_folds(train_rows,n_splits=5):\n    """Balance sentence counts using nearest valid document boundary, without shuffle.\n\n    Only ids/document ids are read. No tags, words, dev or test data select cuts.\n    Tie: earlier boundary. Each remaining fold must retain >= 1 whole document.\n    """\n    if isinstance(n_splits,bool) or not isinstance(n_splits,Integral) or n_splits<2:\n        raise ValueError(\'n_splits must be an integer >= 2.\')\n    starts=_document_starts(train_rows); n=len(train_rows)\n    if len(starts)<n_splits:\n        raise ValueError(\'Fewer documents than requested folds.\')\n    cut_docs=[0]\n    for i in range(1,n_splits):\n        choices=range(cut_docs[-1]+1,len(starts)-(n_splits-i)+1)\n        cut_docs.append(min(choices,key=lambda j:(abs(starts[j]*n_splits-n*i),starts[j])))\n    bounds=[starts[j] for j in cut_docs]+[n]\n    folds=[]\n    for fold,(start,end) in enumerate(zip(bounds,bounds[1:])):\n        valid=list(range(start,end)); train=list(range(start))+list(range(end,n))\n        train_docs={int(train_rows[i][\'document_id\']) for i in train}\n        valid_docs={int(train_rows[i][\'document_id\']) for i in valid}\n        folds.append(dict(fold=fold,train_indices=train,validation_indices=valid,\n            validation_start=start,validation_end_exclusive=end,train_sentences=len(train),validation_sentences=len(valid),\n            train_documents=len(train_docs),validation_documents=len(valid_docs),shared_documents=len(train_docs & valid_docs)))\n    plan=dict(split=\'train\',n_splits=int(n_splits),shuffle=False,sentence_count=n,document_count=len(starts),\n        algorithm=\'Nearest equal-sentence target at a document boundary; earlier tie; preserve >=1 document per remaining fold.\',\n        rows=[dict(index=i,sentence_id=str(r[\'id\']),document_id=int(r[\'document_id\'])) for i,r in enumerate(train_rows)],folds=folds)\n    verify_fold_plan(plan,train_rows)\n    return plan\n\n\ndef verify_fold_plan(plan,train_rows):\n    n=len(train_rows)\n    if plan[\'split\']!=\'train\' or plan[\'sentence_count\']!=n or plan[\'shuffle\'] is not False:\n        raise ValueError(\'Fold plan has the wrong source split/count/shuffle rule.\')\n    expected=[dict(index=i,sentence_id=str(r[\'id\']),document_id=int(r[\'document_id\'])) for i,r in enumerate(train_rows)]\n    if plan[\'rows\']!=expected or len({r[\'sentence_id\'] for r in expected})!=n:\n        raise ValueError(\'Fold plan ids or order differ from training data.\')\n    if plan[\'document_count\']!=len(_document_starts(train_rows)) or len(plan[\'folds\'])!=plan[\'n_splits\']:\n        raise ValueError(\'Document/fold count mismatch.\')\n    all_indices=set(range(n)); held_out=[]; last_end=0\n    for k,fold in enumerate(plan[\'folds\']):\n        start,end=fold[\'validation_start\'],fold[\'validation_end_exclusive\']\n        if fold[\'fold\']!=k or start!=last_end or not start<end<=n:\n            raise ValueError(\'Validation blocks must be consecutive, disjoint and nonempty.\')\n        valid=fold[\'validation_indices\']; train=fold[\'train_indices\']\n        if valid!=list(range(start,end)) or train!=sorted(all_indices-set(valid)):\n            raise ValueError(\'Fold indices are not an exact train/validation partition.\')\n        td={train_rows[i][\'document_id\'] for i in train}; vd={train_rows[i][\'document_id\'] for i in valid}\n        if td & vd or fold[\'shared_documents\']!=0:\n            raise ValueError(\'A document leaks across the fold boundary.\')\n        if (len(train),len(valid),len(td),len(vd))!=(fold[\'train_sentences\'],fold[\'validation_sentences\'],fold[\'train_documents\'],fold[\'validation_documents\']):\n            raise ValueError(\'Fold summary counts disagree with indices.\')\n        held_out.extend(valid); last_end=end\n    if held_out!=list(range(n)):\n        raise ValueError(\'Each training sentence must be held out exactly once.\')\n    return dict(valid=True,held_out_once=n,shared_documents=0,n_splits=plan[\'n_splits\'])\n\n\ndef build_protocol(accepted_data,label_mapping,fold_plans,source_hashes):\n    """No runtime paths/timestamps: identical inputs produce the same contract hash."""\n    labels=label_mapping[\'label_names\']\n    return {\n        \'version\':PROTOCOL_VERSION,\'status\':\'frozen experimental design; model-specific grids must be registered before their search\',\n        \'dataset\':{\'source\':accepted_data[\'source\'][\'actual_source\'],\'revision\':accepted_data[\'source\'][\'revision\'],\n                   \'source_files\':accepted_data[\'source\'][\'source_files\'],\'audit\':accepted_data[\'audit\']},\n        \'label_names\':labels,\'metric_contract\':METRIC_CONTRACT,\'prediction_schema\':SCHEMA,\n        \'fold_plan_sha256\':{name:fingerprint(plan) for name,plan in fold_plans.items()},\n        \'public_source_sha256\':source_hashes,\n        \'seeds\':{\'classical\':42,\'random_search\':42,\'transformer\':SEEDS,\'bootstrap\':BOOTSTRAP_SEED,\'representation_sampling\':42},\n        \'split_usage\':{\n            \'train\':\'All fitting. Classical hyperparameters use the saved 5 contiguous document-safe training folds.\',\n            \'validation\':\'Compare methods/features; choose Transformer hyperparameters/checkpoints/early stopping; perform all error analysis.\',\n            \'test\':\'Only after final settings/checkpoints and comparison list are frozen. Score each final checkpoint once; save predictions for all later uncertainty analysis. No tuning or error-driven changes.\'},\n        \'classical\':{\'primary_folds\':5,\'selection\':\'Mean of held-out fold micro entity F1, not token accuracy; report fold std (ddof=1).\',\n            \'tie_break\':\'First configuration in the pre-registered simpler-to-more-complex search order.\',\n            \'crf_fallback\':{\'folds\':3,\'search\':\'randomized, 10 configurations, seed 42\',\n                \'trigger\':\'A training-only timing pilot predicts >4 CPU hours for the registered 5-fold search; log estimate and switch before viewing search scores.\',\n                \'report\':\'Identify fallback explicitly; no switching based on CV/dev/test performance.\'},\n            \'leakage_rule\':\'Fit vectorizers, learned vocabularies, scalers, SVD/PCA, K-means and all other learned transforms only on the fit portion of each fold. Refit selected transforms/model on train, never dev/test.\',\n            \'features\':\'F3 gold POS/chunk is research-only; deployment variants use raw-text-available features.\'},\n        \'transformer\':{\'encoders\':[\'bert-base-cased\',\'roberta-base\'],\'max_length\':256,\n            \'preprocessing\':\'Accepted Step 3 first-subword supervision; reconstruct original sentences before scoring.\',\n            \'selection\':\'Tune on validation with seed 42, freeze settings, then repeat the chosen setting for seeds 42/52/62. Each seed selects its checkpoint by validation F1.\',\n            \'report\':\'Report all three seeds and mean/sample std (ddof=1). Never select the best test seed.\',\n            \'primary_comparison_seed\':42,\'full_kfold\':False,\n            \'reason\':\'Full K-fold fine-tuning is too costly; held-out development selection plus independent seed repeats.\'},\n        \'uncertainty\':{\'n_resamples\':2000,\'confidence\':0.95,\'seed\':BOOTSTRAP_SEED,\'method\':\'sentence percentile\',\n            \'paired_rule\':\'Same original sentence indices for A and B; resample saved predictions, no refitting.\',\n            \'preplanned_primary_pairs\':[\'selected deployable CRF vs BERT seed 42\',\'BERT seed 42 vs RoBERTa seed 42\'],\n            \'report\':\'B-minus-A intervals on corpus micro F1. Per-type comparisons exploratory; intervals are not multiple-comparison corrected or training-variance intervals.\'},\n        \'change_control\':\'Never overwrite this v1 contract. Any necessary design change gets a new version and rationale before model selection/test scoring; keep the old contract. Register exact search spaces/budgets before search and final model settings before test.\',\n        \'step4_scope\':\'Real data integrity/fold generation only. Metric/bootstrap/prediction-file demonstrations are synthetic. No real model predictions or test scores are generated.\'}\n\n\ndef protocol_table(protocol):\n    rows=[]\n    for split,rule in protocol[\'split_usage\'].items():\n        rows.append(dict(component=\'data\',setting=split,value=str(protocol[\'dataset\'][\'audit\'][\'splits\'][split][\'rows\']),rule=rule))\n    for key,value in protocol[\'seeds\'].items():\n        rows.append(dict(component=\'seed\',setting=key,value=str(value),rule=\'Fixed before fitting/selection.\'))\n    for name,rule,value in [\n        (\'metric\',METRIC_CONTRACT[\'primary\'],\'strict entity micro P/R/F1\'),\n        (\'prediction_decode\',METRIC_CONTRACT[\'prediction\'],\'orphan I-X -> B-X\'),\n        (\'classical_cv\',protocol[\'classical\'][\'selection\'],\'5 contiguous document-safe folds\'),\n        (\'crf_fallback\',protocol[\'classical\'][\'crf_fallback\'][\'trigger\'],\'3 folds, randomized 10 configurations\'),\n        (\'transform_fit\',protocol[\'classical\'][\'leakage_rule\'],\'fit inside each training fold\'),\n        (\'transformer\',protocol[\'transformer\'][\'selection\'],\'3 seeds; validation selection\'),\n        (\'bootstrap\',METRIC_CONTRACT[\'bootstrap\'],\'2000 draws, 95%, seed 2026\'),\n        (\'paired\',METRIC_CONTRACT[\'paired\'],\'same resampled sentences\'),\n        (\'test\',protocol[\'split_usage\'][\'test\'],\'once per frozen final checkpoint\')]:\n        rows.append(dict(component=\'protocol\',setting=name,value=value,rule=rule))\n    return rows\n\n\ndef freeze_protocol(project_root,protocol,fold_plans):\n    directory=Path(project_root)/\'config\'/PROTOCOL_VERSION\n    payloads={\'protocol.json\':canonical_bytes(protocol), **{name+\'.json\':canonical_bytes(plan) for name,plan in fold_plans.items()}}\n    if directory.exists():\n        if {p.name for p in directory.iterdir()}!=set(payloads) or any((directory/name).read_bytes()!=content for name,content in payloads.items()):\n            raise ValueError(\'A different/incomplete v1 protocol already exists. Keep it; investigate or create an explicitly versioned revision.\')\n        reused=True\n    else:\n        directory.mkdir(parents=True,exist_ok=False)\n        for name,content in payloads.items():\n            with (directory/name).open(\'xb\') as stream: stream.write(content)\n        reused=False\n    if any((directory/name).read_bytes()!=content for name,content in payloads.items()):\n        raise ValueError(\'Frozen protocol read-back mismatch.\')\n    return dict(path=str(directory),protocol_sha256=fingerprint(protocol),reused_identical=reused,\n        files=[dict(path=name,size_bytes=len(content),sha256=hashlib.sha256(content).hexdigest()) for name,content in payloads.items()],read_back_verified=True)\n', 'local_backup.py': '"""Verified, portable NER backups. Uses local files and the Python standard library."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nFORMAT = \'cs5489-ner-local-backup-v1\'\nALLOWED_FOLDERS = {\'config\', \'logs\', \'src\', \'results\', \'figures\', \'predictions\', \'notebooks\'}\nALLOWED_ROOT_FILES = {\'README.md\', \'requirements.txt\', \'requirements-step0.txt\'}\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _relative_path(value):\n    if not isinstance(value, str) or not value or \'\\\\\' in value or \':\' in value:\n        raise ValueError(\'Invalid backup member path.\')\n    parts = value.split(\'/\')\n    if any(part in (\'\', \'.\', \'..\') or part.startswith(\'.\') for part in parts):\n        raise ValueError(\'Unsafe backup member path.\')\n    if PurePosixPath(value).is_absolute():\n        raise ValueError(\'Absolute backup member paths are forbidden.\')\n    return PurePosixPath(value)\n\n\ndef _is_project_output(relative, include_checkpoints):\n    parts = relative.parts\n    if any(part.startswith(\'.\') or part == \'__pycache__\' for part in parts):\n        return False\n    filename = relative.name.lower()\n    if (\'credential\' in filename or \'oauth\' in filename or filename.startswith(\'client_secret\') or\n            filename in {\'token.json\', \'tokens.json\'} or\n            any(secret in filename for secret in (\'access_token\', \'refresh_token\', \'id_token\')) or\n            filename.endswith((\'.zip\', \'.pyc\', \'.pem\', \'.key\'))):\n        return False\n    if len(parts) == 1:\n        return relative.name in ALLOWED_ROOT_FILES\n    return parts[0] in ALLOWED_FOLDERS or (include_checkpoints and parts[0] == \'checkpoints\')\n\n\ndef verify_backup(archive_file, max_total_bytes=20 * 1024**3):\n    """Check structure, file sizes and hashes before accepting a local ZIP.\n\n    Hashes detect corruption; they do not establish who created an archive.\n    Restore only backups from a source you trust.\n    """\n    with zipfile.ZipFile(archive_file) as archive:\n        members = archive.infolist()\n        names = [item.filename for item in members]\n        if len(names) != len(set(names)) or \'manifest.json\' not in names:\n            raise ValueError(\'Backup has duplicate members or no manifest.\')\n        if sum(item.file_size for item in members) > max_total_bytes:\n            raise ValueError(\'Backup exceeds the configured uncompressed-size limit.\')\n        for item in members:\n            _relative_path(item.filename)\n            if stat.S_ISLNK(item.external_attr >> 16) or item.is_dir():\n                raise ValueError(\'Links and directory entries are not valid backup members.\')\n        if archive.getinfo(\'manifest.json\').file_size > 10 * 1024**2:\n            raise ValueError(\'Backup manifest is too large.\')\n        manifest = json.loads(archive.read(\'manifest.json\'))\n        if manifest.get(\'format\') != FORMAT or not isinstance(manifest.get(\'files\'), list):\n            raise ValueError(\'Unknown backup format.\')\n        expected_names = {\'manifest.json\'}\n        paths = set()\n        for record in manifest[\'files\']:\n            relative = _relative_path(record[\'path\'])\n            if relative.as_posix() in paths:\n                raise ValueError(\'Duplicate path in backup manifest.\')\n            paths.add(relative.as_posix())\n            member_name = \'project/\' + relative.as_posix()\n            expected_names.add(member_name)\n            if member_name not in names:\n                raise ValueError(\'A manifest file is missing from the ZIP.\')\n            info = archive.getinfo(member_name)\n            if info.file_size != record[\'size_bytes\']:\n                raise ValueError(f\'Backup size check failed: {relative}\')\n            digest = hashlib.sha256()\n            with archive.open(info) as stream:\n                for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n                    digest.update(chunk)\n            if digest.hexdigest() != record[\'sha256\']:\n                raise ValueError(f\'Backup checksum check failed: {relative}\')\n        if set(names) != expected_names:\n            raise ValueError(\'Backup contains files not listed in the manifest.\')\n        # Reading every member to EOF above also checks each ZIP CRC.\n    return manifest\n\n\ndef create_backup(project_root, export_dir, step, revision, include_checkpoints=False):\n    """Archive selected project outputs and verify the completed ZIP by reading it."""\n    root = Path(project_root).resolve(strict=True)\n    export = Path(export_dir).resolve()\n    if not root.is_dir() or export.is_relative_to(root):\n        raise ValueError(\'Use an existing project directory and an export directory outside it.\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]+\', step):\n        raise ValueError(\'Use a simple step name, e.g. step0.\')\n    export.mkdir(parents=True, exist_ok=True)\n    records, selected = [], []\n    folders = ALLOWED_FOLDERS | ({\'checkpoints\'} if include_checkpoints else set())\n    candidates = [root / name for name in ALLOWED_ROOT_FILES if (root / name).is_file()]\n    for name in sorted(folders):\n        folder = root / name\n        if folder.is_symlink():\n            raise ValueError(f\'Refusing to back up a linked project directory: {name}\')\n        if folder.is_dir():\n            candidates.extend(folder.rglob(\'*\'))\n    for path in sorted(candidates):\n        relative = path.relative_to(root)\n        if not _is_project_output(relative, include_checkpoints):\n            continue\n        if path.is_symlink() or any((root / Path(*relative.parts[:i])).is_symlink()\n                                    for i in range(1, len(relative.parts))):\n            raise ValueError(f\'Refusing to back up a link: {relative}\')\n        if not path.is_file():\n            continue\n        path.resolve(strict=True).relative_to(root)\n        records.append({\'path\': relative.as_posix(), \'size_bytes\': path.stat().st_size,\n                        \'sha256\': sha256_file(path)})\n        selected.append((path, relative))\n    if not records:\n        raise ValueError(\'No project outputs were selected for backup.\')\n    created = datetime.now(timezone.utc)\n    manifest = {\'format\': FORMAT, \'step\': step, \'revision\': revision,\n                \'created_at_utc\': created.isoformat(), \'files\': records,\n                \'includes_checkpoints\': include_checkpoints}\n    name = f\'NER_{step}_{created:%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:8]}.zip\'\n    output = export / name\n    partial = output.with_suffix(\'.partial\')\n    with zipfile.ZipFile(partial, \'x\', compression=zipfile.ZIP_DEFLATED,\n                         compresslevel=2, strict_timestamps=False) as archive:\n        archive.writestr(\'manifest.json\', json.dumps(manifest, indent=2))\n        for path, relative in selected:\n            archive.write(path, arcname=\'project/\' + relative.as_posix())\n    verify_backup(partial)\n    if output.exists():\n        raise FileExistsError(output)\n    partial.rename(output)\n    return {\'path\': str(output), \'name\': output.name, \'size_bytes\': output.stat().st_size,\n            \'sha256\': sha256_file(output), \'file_count\': len(records),\n            \'archive_verified\': True, \'download_confirmed\': False}\n\n\ndef restore_backup(archive_file, destination):\n    """Restore verified bytes into a new directory without executing any code."""\n    manifest = verify_backup(archive_file)\n    target = Path(destination).resolve()\n    if target.exists():\n        raise FileExistsError(\'Choose a new, empty destination name for restoration.\')\n    target.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(archive_file) as archive:\n        for record in manifest[\'files\']:\n            relative = _relative_path(record[\'path\'])\n            path = target.joinpath(*relative.parts)\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(\'project/\' + relative.as_posix()) as source, path.open(\'xb\') as output:\n                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)\n            if sha256_file(path) != record[\'sha256\']:\n                raise ValueError(f\'Restored file checksum failed: {relative}\')\n    return {\'project_root\': str(target), \'step\': manifest[\'step\'], \'file_count\': len(manifest[\'files\'])}\n', 'metrics.py': '"""Versioned word-level NER evaluation. No training or data downloads."""\nimport math\nfrom numbers import Integral\n\nTYPES = (\'PER\', \'ORG\', \'LOC\', \'MISC\')\nLABELS = {\'O\'} | {f\'{prefix}-{kind}\' for prefix in (\'B\', \'I\') for kind in TYPES}\nMETRIC_CONTRACT = {\n    \'version\': \'ner-entity-metrics-v1\',\n    \'primary\': \'Micro entity precision/recall/F1; exact start, exclusive end and type, within each original sentence.\',\n    \'seqeval\': \'1.2.2, mode=strict, scheme=IOB2, zero_division=0; independently checked against span counts.\',\n    \'gold\': \'Original metadata-decoded IOB2; invalid gold or mismatched sentence/token lengths raises an error.\',\n    \'prediction\': \'Before all scoring, an I-X not preceded by B-X/I-X becomes B-X. Other tags unchanged. Same fixed rule for every system; preserve raw tags and count repairs.\',\n    \'rationale\': \'Matches development-document Step 9 orphan-I extraction rule; does not inspect gold labels. Boundary/type matching remains exact.\',\n    \'missing_types\': \'Always output PER/ORG/LOC/MISC; undefined precision, recall or F1 is 0, with explicit support and denominator counts.\',\n    \'bootstrap\': \'2000 sentence-level draws with replacement, 95% percentile intervals, NumPy default_rng seed 2026; sum TP/FP/FN then compute corpus metrics.\',\n    \'paired\': \'B minus A on identical sentence indices per draw; require identical ids, tokens and gold. An interval excluding zero is descriptive evidence, not a calibrated p-value.\',\n    \'uncertainty_scope\': \'Conditional on fixed trained models and sentence resampling; does not include training-seed variance or within-document dependence.\',\n    \'scale\': \'All metrics and confidence limits are fractions in [0,1]; paired differences in [-1,1].\',\n}\n\n\ndef _sentences(value, name):\n    if not isinstance(value, (list, tuple)) or not value:\n        raise ValueError(name + \' must be a nonempty sequence of original sentences.\')\n    result = []\n    for row in value:\n        if not isinstance(row, (list, tuple)) or not row or any(not isinstance(t, str) or t not in LABELS for t in row):\n            raise ValueError(name + \' contains an empty sentence, non-string or unknown tag.\')\n        result.append(list(row))\n    return result\n\n\ndef entity_spans(tags):\n    """Strict IOB2 -> set of (start, exclusive_end, type). Never crosses sentences."""\n    if not isinstance(tags, (list, tuple)) or any(not isinstance(t, str) or t not in LABELS for t in tags):\n        raise ValueError(\'Expected known IOB2 tag strings.\')\n    spans, start, kind = set(), None, None\n    for i, tag in enumerate([*tags, \'O\']):\n        if tag.startswith(\'I-\'):\n            if start is None or kind != tag[2:]:\n                raise ValueError(\'Invalid IOB2 continuation in gold/canonical tags.\')\n        else:\n            if start is not None:\n                spans.add((start, i, kind))\n                start = kind = None\n            if tag.startswith(\'B-\'):\n                start, kind = i, tag[2:]\n    return spans\n\n\ndef canonicalize_predictions(y_pred):\n    rows = _sentences(y_pred, \'predictions\')\n    repairs = 0\n    output = []\n    for row in rows:\n        previous, canonical = \'O\', []\n        for tag in row:\n            if tag.startswith(\'I-\') and previous not in (\'B-\' + tag[2:], tag):\n                tag = \'B-\' + tag[2:]\n                repairs += 1\n            canonical.append(tag)\n            previous = tag\n        output.append(canonical)\n    return output, repairs\n\n\ndef sentence_counts(y_true, y_pred):\n    """Return counts[N, 4, 3] for TP/FP/FN, plus validated/canonical tag rows."""\n    import numpy as np\n    gold = _sentences(y_true, \'gold\')\n    pred, repairs = canonicalize_predictions(y_pred)\n    if len(gold) != len(pred) or any(len(a) != len(b) for a,b in zip(gold,pred)):\n        raise ValueError(\'Gold/prediction sentence or word counts differ.\')\n    counts = np.zeros((len(gold), len(TYPES), 3), dtype=np.int64)\n    for i, (a,b) in enumerate(zip(gold,pred)):\n        truth, guess = entity_spans(a), entity_spans(b)\n        for j, kind in enumerate(TYPES):\n            actual = {span for span in truth if span[2] == kind}\n            found = {span for span in guess if span[2] == kind}\n            counts[i,j] = (len(actual & found), len(found-actual), len(actual-found))\n    return counts, gold, pred, repairs\n\n\ndef _scores(counts):\n    import numpy as np\n    counts = np.asarray(counts)\n    tp, fp, fn = (counts[...,i] for i in range(3))\n    def divide(a,b):\n        return np.divide(a,b,out=np.zeros_like(a,dtype=float),where=b!=0)\n    return np.stack((divide(tp,tp+fp), divide(tp,tp+fn), divide(2*tp,2*tp+fp+fn)),axis=-1)\n\n\ndef _count_report(counts):\n    tp, fp, fn = map(int, counts)\n    p,r,f = map(float, _scores(counts))\n    return dict(tp=tp, fp=fp, fn=fn, support=tp+fn, predicted=tp+fp, precision=p, recall=r, f1=f)\n\n\ndef evaluate_ner(y_true, y_pred):\n    """Primary project scorer: strict seqeval on fixed, gold-independent IOB2 decoding."""\n    import numpy as np\n    from seqeval.metrics import precision_score, recall_score, f1_score\n    from seqeval.scheme import IOB2\n    counts, gold, pred, repairs = sentence_counts(y_true,y_pred)\n    totals = counts.sum(axis=0)\n    overall = _count_report(totals.sum(axis=0))\n    per_type = {kind:_count_report(totals[i]) for i,kind in enumerate(TYPES)}\n    # Filter by type to verify every metric even when one type has no support.\n    for kind, target in [(\'overall\', overall), *per_type.items()]:\n        true_rows = gold if kind==\'overall\' else [[t if t==\'O\' or t[2:]==kind else \'O\' for t in row] for row in gold]\n        pred_rows = pred if kind==\'overall\' else [[t if t==\'O\' or t[2:]==kind else \'O\' for t in row] for row in pred]\n        observed = [fn(true_rows,pred_rows,average=\'micro\',mode=\'strict\',scheme=IOB2,zero_division=0)\n                    for fn in (precision_score,recall_score,f1_score)]\n        if not np.allclose(observed,[target[k] for k in (\'precision\',\'recall\',\'f1\')],atol=1e-12,rtol=0):\n            raise ValueError(\'seqeval differs from independent exact-span counts: \' + kind)\n    return dict(contract=METRIC_CONTRACT[\'version\'],overall=overall,per_type=per_type,\n                sentences=len(gold),words=sum(map(len,gold)),prediction_iob2_repairs=repairs,\n                seqeval_crosscheck_passed=True)\n\n\ndef entity_f1(y_true,y_pred):\n    """CV scoring function; call with lists of original sentence label lists."""\n    return evaluate_ner(y_true,y_pred)[\'overall\'][\'f1\']\n\n\ndef estimator_entity_f1(estimator, X, y_true):\n    """scikit-learn callable scorer. Estimator.predict(X) must return sentence lists."""\n    def nested_lists(value):\n        rows=list(value)\n        if any(isinstance(row,str) for row in rows):\n            raise ValueError(\'CV scorer requires sentence lists, not flattened token labels.\')\n        return [list(row) for row in rows]\n    return entity_f1(nested_lists(y_true),nested_lists(estimator.predict(X)))\n\n\ndef _bootstrap_args(n_resamples,confidence,seed):\n    if isinstance(n_resamples,bool) or not isinstance(n_resamples,Integral) or n_resamples<2:\n        raise ValueError(\'n_resamples must be an integer >= 2.\')\n    if isinstance(seed,bool) or not isinstance(seed,Integral) or seed<0:\n        raise ValueError(\'seed must be a nonnegative integer.\')\n    if isinstance(confidence,bool) or not isinstance(confidence,(int,float)) or not math.isfinite(confidence) or not 0<confidence<1:\n        raise ValueError(\'confidence must be between zero and one.\')\n\n\ndef _with_overall(counts):\n    import numpy as np\n    return np.concatenate((counts.sum(axis=1,keepdims=True),counts),axis=1)\n\n\ndef _intervals(samples, point, confidence):\n    import numpy as np\n    alpha=(1-confidence)/2\n    limits=np.quantile(samples,[alpha,1-alpha],axis=0,method=\'linear\')\n    return {kind:{metric:dict(estimate=float(point[i,j]),low=float(limits[0,i,j]),high=float(limits[1,i,j]))\n             for j,metric in enumerate((\'precision\',\'recall\',\'f1\'))}\n            for i,kind in enumerate((\'overall\',*TYPES))}\n\n\ndef bootstrap_ci(y_true,y_pred,n_resamples=2000,confidence=0.95,seed=2026):\n    """Sentence percentile bootstrap; resample counts, not token rows or sentence F1s."""\n    import numpy as np\n    _bootstrap_args(n_resamples,confidence,seed)\n    point_report=evaluate_ner(y_true,y_pred)\n    counts=_with_overall(sentence_counts(y_true,y_pred)[0])\n    rng=np.random.default_rng(seed)\n    samples=np.empty((n_resamples,5,3)); empty=np.zeros(5,dtype=np.int64)\n    for b in range(n_resamples):\n        total=counts[rng.integers(0,len(counts),size=len(counts))].sum(axis=0)\n        samples[b]=_scores(total)\n        empty += (total.sum(axis=1)==0)\n    return dict(method=\'sentence_percentile\',seed=int(seed),n_resamples=int(n_resamples),confidence=confidence,\n        unit=\'original sentence\',point=point_report,intervals=_intervals(samples,_scores(counts.sum(axis=0)),confidence),\n        zero_f1_denominator_draws=dict(zip((\'overall\',*TYPES),map(int,empty))),\n        limitation=METRIC_CONTRACT[\'uncertainty_scope\'])\n\n\ndef paired_bootstrap(y_true,y_pred_a,y_pred_b,n_resamples=2000,confidence=0.95,seed=2026):\n    """B-minus-A percentile intervals using exactly the same indices in each draw.\n\n    Arrays must already be aligned to the same original sentences. For saved files,\n    use prediction_io.compare_prediction_files to validate ids/tokens/gold first.\n    """\n    import numpy as np\n    _bootstrap_args(n_resamples,confidence,seed)\n    point_a,point_b=evaluate_ner(y_true,y_pred_a),evaluate_ner(y_true,y_pred_b)\n    a=_with_overall(sentence_counts(y_true,y_pred_a)[0]); b=_with_overall(sentence_counts(y_true,y_pred_b)[0])\n    rng=np.random.default_rng(seed); samples=np.empty((n_resamples,5,3))\n    for i in range(n_resamples):\n        selected=rng.integers(0,len(a),size=len(a))\n        samples[i]=_scores(b[selected].sum(axis=0))-_scores(a[selected].sum(axis=0))\n    delta=_scores(b.sum(axis=0))-_scores(a.sum(axis=0))\n    intervals=_intervals(samples,delta,confidence)\n    return dict(method=\'paired_sentence_percentile\',direction=\'B minus A\',seed=int(seed),n_resamples=int(n_resamples),\n        confidence=confidence,point_a=point_a,point_b=point_b,intervals=intervals,\n        overall_f1_interval_excludes_zero=bool(intervals[\'overall\'][\'f1\'][\'low\']>0 or intervals[\'overall\'][\'f1\'][\'high\']<0),\n        interpretation=\'Percentile difference interval; no p-value computed. Predeclare comparisons; per-type intervals are not multiplicity-adjusted.\',\n        limitation=METRIC_CONTRACT[\'uncertainty_scope\'])\n\n\ndef toy_checks():\n    """Real seqeval checks on fabricated tags only; not model performance."""\n    cases=[\n        (\'exact\',[[\'B-PER\',\'I-PER\',\'O\']],[[\'B-PER\',\'I-PER\',\'O\']],(1,0,0)),\n        (\'Jane_only\',[[\'B-PER\',\'I-PER\']],[[\'B-PER\',\'O\']],(0,1,1)),\n        (\'wrong_type\',[[\'B-PER\',\'I-PER\']],[[\'B-ORG\',\'I-ORG\']],(0,1,1)),\n        (\'missed\',[[\'B-LOC\']],[[\'O\']],(0,0,1)),\n        (\'spurious\',[[\'O\']],[[\'B-MISC\']],(0,1,0)),\n        (\'all_O\',[[\'O\',\'O\']],[[\'O\',\'O\']],(0,0,0)),\n        (\'adjacent_entities\',[[\'B-PER\',\'B-PER\']],[[\'B-PER\',\'B-PER\']],(2,0,0)),\n        (\'sentence_boundary\',[[\'B-PER\'],[\'B-PER\']],[[\'B-PER\'],[\'I-PER\']],(2,0,0)),\n        (\'orphan_I_false_positive\',[[\'O\']],[[\'I-PER\']],(0,1,0)),\n        (\'type_switch_I\',[[\'B-PER\',\'B-ORG\']],[[\'B-PER\',\'I-ORG\']],(2,0,0)),\n    ]\n    results=[]\n    for name,gold,pred,expected in cases:\n        result=evaluate_ner(gold,pred)\n        if tuple(result[\'overall\'][k] for k in (\'tp\',\'fp\',\'fn\'))!=expected:\n            raise ValueError(\'Toy metric mismatch: \'+name)\n        results.append(dict(case=name,synthetic_only=True,**result))\n    return results\n', 'prediction_io.py': '"""Shared prediction JSONL contract, integrity checks and paired-file comparison."""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nfrom metrics import LABELS, METRIC_CONTRACT, canonicalize_predictions, entity_spans, paired_bootstrap\n\nSCHEMA = {\n    \'version\':\'ner-word-predictions-v1\',\n    \'filename\':\'<system>_<split>.jsonl plus .meta.json sidecar\',\n    \'fields\':{\'sentence_id\':\'unique string; original dataset id\', \'tokens\':\'original word strings in order\',\n              \'gold_tags\':\'metadata-decoded, valid IOB2 tags\', \'pred_tags\':\'canonical IOB2 predictions\',\n              \'raw_pred_tags\':\'unaltered model tag strings before the fixed orphan-I rule\'},\n    \'alignment\':\'Exactly one row per original sentence, one tag per original word. Reassemble Transformer chunks first.\',\n    \'sidecar\':\'schema/metric version, system, split, protocol hash, counts and JSONL SHA-256; no overwrite.\',\n    \'test_use\':\'Write final test predictions only after final settings are frozen; re-use saved files for bootstrap, never retrain during resampling.\',\n}\n\n\ndef _digest(content):\n    return hashlib.sha256(content).hexdigest()\n\n\ndef validate_records(records, expected_rows=None, label_names=None):\n    if not isinstance(records,list) or not records:\n        raise ValueError(\'Prediction records must be a nonempty list.\')\n    seen=set()\n    for row in records:\n        if not isinstance(row,dict) or set(row)!=set(SCHEMA[\'fields\']):\n            raise ValueError(\'Prediction row fields do not match the versioned schema.\')\n        identity=row[\'sentence_id\']; tokens=row[\'tokens\']\n        if not isinstance(identity,str) or not identity or identity in seen:\n            raise ValueError(\'Missing, non-string or duplicate sentence id.\')\n        seen.add(identity)\n        if not isinstance(tokens,list) or not tokens or any(not isinstance(t,str) or not t for t in tokens):\n            raise ValueError(\'Invalid original token list.\')\n        for field in (\'gold_tags\',\'pred_tags\',\'raw_pred_tags\'):\n            if not isinstance(row[field],list) or len(row[field])!=len(tokens):\n                raise ValueError(\'Prediction/gold/token length mismatch.\')\n        entity_spans(row[\'gold_tags\']); entity_spans(row[\'pred_tags\'])\n        canonical,_=canonicalize_predictions([row[\'raw_pred_tags\']])\n        if canonical[0]!=row[\'pred_tags\']:\n            raise ValueError(\'Saved prediction differs from canonical raw prediction.\')\n    if expected_rows is not None:\n        if label_names is None or set(label_names)!=LABELS or len(label_names)!=len(LABELS):\n            raise ValueError(\'Expected-data validation requires the metadata label order.\')\n        if len(expected_rows)!=len(records):\n            raise ValueError(\'Prediction file misses or adds source sentences.\')\n        for actual,source in zip(records,expected_rows):\n            if actual[\'sentence_id\']!=str(source[\'id\']) or actual[\'tokens\']!=source[\'tokens\'] or actual[\'gold_tags\']!=[label_names[v] for v in source[\'ner_tags\']]:\n                raise ValueError(\'Prediction file ids/tokens/gold/order differ from source data.\')\n    return dict(sentences=len(records),words=sum(len(r[\'tokens\']) for r in records))\n\n\ndef write_predictions(directory,system,split,source_rows,raw_predictions,label_names,protocol_sha256):\n    if not re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9_-]*\',system) or split not in (\'train\',\'validation\',\'test\',\'synthetic\'):\n        raise ValueError(\'Use a simple system name and known split.\')\n    if not re.fullmatch(r\'[0-9a-f]{64}\',protocol_sha256):\n        raise ValueError(\'Provide the frozen protocol SHA-256.\')\n    if len(source_rows)!=len(raw_predictions):\n        raise ValueError(\'Prediction/source sentence counts differ.\')\n    canonical,repairs=canonicalize_predictions(raw_predictions)\n    records=[dict(sentence_id=str(row[\'id\']),tokens=list(row[\'tokens\']),\n        gold_tags=[label_names[v] for v in row[\'ner_tags\']],pred_tags=pred,raw_pred_tags=list(raw))\n        for row,raw,pred in zip(source_rows,raw_predictions,canonical)]\n    counts=validate_records(records,source_rows,label_names)\n    path=Path(directory)/(system+\'_\'+split+\'.jsonl\')\n    sidecar=path.with_suffix(\'.meta.json\')\n    if path.exists() or sidecar.exists():\n        raise FileExistsError(\'Do not overwrite predictions; use a unique system/run name.\')\n    content=\'\'.join(json.dumps(r,ensure_ascii=False,separators=(\',\',\':\'))+\'\\n\' for r in records).encode(\'utf-8\')\n    metadata=dict(schema=SCHEMA[\'version\'],metric_contract=METRIC_CONTRACT[\'version\'],system=system,split=split,\n                  protocol_sha256=protocol_sha256,sha256=_digest(content),prediction_iob2_repairs=repairs,**counts)\n    path.parent.mkdir(parents=True,exist_ok=True)\n    with path.open(\'xb\') as stream: stream.write(content)\n    with sidecar.open(\'x\',encoding=\'utf-8\') as stream: json.dump(metadata,stream,ensure_ascii=False,indent=2)\n    restored,restored_meta=load_predictions(path,source_rows,label_names)\n    if restored!=records or restored_meta!=metadata:\n        raise ValueError(\'Prediction JSONL save/read-back mismatch.\')\n    return dict(path=str(path),metadata_path=str(sidecar),**metadata,read_back_verified=True)\n\n\ndef load_predictions(path,expected_rows=None,label_names=None):\n    path=Path(path)\n    metadata=json.loads(path.with_suffix(\'.meta.json\').read_text(encoding=\'utf-8\'))\n    content=path.read_bytes()\n    if metadata.get(\'schema\')!=SCHEMA[\'version\'] or metadata.get(\'metric_contract\')!=METRIC_CONTRACT[\'version\'] or metadata.get(\'sha256\')!=_digest(content):\n        raise ValueError(\'Prediction version or file checksum mismatch.\')\n    records=[json.loads(line) for line in content.decode(\'utf-8\').splitlines()]\n    counts=validate_records(records,expected_rows,label_names)\n    repairs=canonicalize_predictions([r[\'raw_pred_tags\'] for r in records])[1]\n    if any(metadata.get(k)!=v for k,v in counts.items()) or metadata.get(\'prediction_iob2_repairs\')!=repairs:\n        raise ValueError(\'Prediction sidecar counts mismatch.\')\n    if not re.fullmatch(r\'[0-9a-f]{64}\',metadata.get(\'protocol_sha256\',\'\')):\n        raise ValueError(\'Missing protocol fingerprint.\')\n    return records,metadata\n\n\ndef compare_prediction_files(path_a,path_b,**bootstrap_options):\n    a,ma=load_predictions(path_a); b,mb=load_predictions(path_b)\n    if ma[\'split\']!=mb[\'split\'] or ma[\'protocol_sha256\']!=mb[\'protocol_sha256\'] or len(a)!=len(b):\n        raise ValueError(\'Paired files must have the same split, protocol and sentence count.\')\n    for x,y in zip(a,b):\n        if any(x[k]!=y[k] for k in (\'sentence_id\',\'tokens\',\'gold_tags\')):\n            raise ValueError(\'Paired files have different ids/tokens/gold/order.\')\n    result=paired_bootstrap([r[\'gold_tags\'] for r in a],[r[\'raw_pred_tags\'] for r in a],\n                            [r[\'raw_pred_tags\'] for r in b],**bootstrap_options)\n    result[\'system_a\']=ma[\'system\']; result[\'system_b\']=mb[\'system\']\n    result[\'prediction_file_sha256\']={\'a\':ma[\'sha256\'],\'b\':mb[\'sha256\']}\n    return result\n', 'step1_data.py': '"""Pinned CoNLL-2003 loading and validation. Run real data work in Colab only."""\nimport hashlib\nimport json\nfrom datetime import datetime, timezone\nfrom numbers import Integral\nfrom pathlib import Path\n\nREPO_ID = \'tomaarsen/conll2003\'\nREVISION = \'3a0c0677d3177b6122fceed3239dda7a5de6a514\'\nEXPECTED_ROWS = {\'train\': 14041, \'validation\': 3250, \'test\': 3453}\nTAG_FIELDS = (\'pos_tags\', \'chunk_tags\', \'ner_tags\')\nREQUIRED_FIELDS = (\'id\', \'tokens\', *TAG_FIELDS)\nEXPECTED_NER_LABELS = {\'O\'} | {f\'{p}-{t}\' for p in (\'B\', \'I\') for t in (\'PER\', \'ORG\', \'LOC\', \'MISC\')}\nPARQUET_FILES = {s: f\'data/{s}-00000-of-00001.parquet\' for s in EXPECTED_ROWS}\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.partial\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef load_pinned_dataset(project_root):\n    """Download public data, without auth or remote loading scripts; retain file hashes."""\n    import datasets\n    from huggingface_hub import snapshot_download\n\n    if datasets.__version__ != \'4.0.0\':\n        raise RuntimeError(\'Use datasets==4.0.0; install it and restart the Python session.\')\n    cache = Path(project_root) / \'data\' / \'hf_cache\'\n    snapshot = Path(snapshot_download(\n        repo_id=REPO_ID, repo_type=\'dataset\', revision=REVISION,\n        allow_patterns=[*PARQUET_FILES.values(), \'README.md\'],\n        cache_dir=str(cache / \'hub\'), token=False,\n    ))\n    files = {split: str(snapshot / name) for split, name in PARQUET_FILES.items()}\n    source_files = []\n    for name in [*PARQUET_FILES.values(), \'README.md\']:\n        path = snapshot / name\n        if not path.is_file():\n            raise FileNotFoundError(f\'Pinned source file missing: {name}\')\n        source_files.append({\'path\': name, \'size_bytes\': path.stat().st_size, \'sha256\': file_sha256(path)})\n    dataset = datasets.load_dataset(\'parquet\', data_files=files,\n                                    cache_dir=str(cache / \'arrow\'), token=False)\n    provenance = {\n        \'requested_source\': \'eriktks/conll2003\',\n        \'actual_source\': REPO_ID, \'revision\': REVISION, \'format\': \'parquet\',\n        \'source_url\': f\'https://huggingface.co/datasets/{REPO_ID}/tree/{REVISION}\',\n        \'reason\': \'Development document permits Parquet fallback; datasets 4.0 removes loading scripts.\',\n        \'license_as_stated_by_card\': \'other\',\n        \'source_files\': source_files,\n        \'preprocessing\': \'None: no shuffle, split, filtering, label conversion or token normalization.\',\n        \'parity_scope\': \'Split sizes, metadata, all-row structure/IOB2 and ordered content hashes checked. \'\n                        \'Byte-for-byte equivalence to eriktks/conll2003 has not been established.\',\n        \'recorded_at_utc\': datetime.now(timezone.utc).isoformat(),\n    }\n    return dataset, provenance\n\n\ndef label_names(split):\n    names = {}\n    for field in TAG_FIELDS:\n        feature = split.features[field]\n        values = getattr(getattr(feature, \'feature\', None), \'names\', None)\n        if not values or len(values) != len(set(values)):\n            raise ValueError(f\'{field} must have unique ClassLabel names in feature metadata.\')\n        names[field] = list(values)\n    if set(names[\'ner_tags\']) != EXPECTED_NER_LABELS:\n        raise ValueError(\'NER metadata must contain O and B/I labels for PER, ORG, LOC, MISC.\')\n    return names\n\n\ndef audit_dataset(dataset, expected_rows=None):\n    """Reject corrupt/IOB1 data instead of silently repairing labels or dropping rows.\n\n    expected_rows is injectable for tiny offline fixtures; real runs use EXPECTED_ROWS.\n    Hash input: UTF-8 compact JSON for each core record, decoded tag strings, plus LF.\n    """\n    expected_rows = EXPECTED_ROWS if expected_rows is None else expected_rows\n    if set(dataset) != set(expected_rows):\n        raise ValueError(\'Unexpected dataset splits.\')\n    report = {\'splits\': {}, \'label_names\': None, \'iob2_violations\': 0,\n              \'hash_format\': \'core-fields-decoded-tags-jsonl-utf8-v1\'}\n    for split_name, expected in expected_rows.items():\n        split = dataset[split_name]\n        if len(split) != expected:\n            raise ValueError(f\'{split_name}: expected {expected} rows, received {len(split)}.\')\n        if not set(REQUIRED_FIELDS).issubset(split.column_names):\n            raise ValueError(f\'{split_name}: missing required columns.\')\n        names = label_names(split)\n        if report[\'label_names\'] is None:\n            report[\'label_names\'] = names\n        elif names != report[\'label_names\']:\n            raise ValueError(f\'{split_name}: label metadata differs between splits.\')\n        has_documents = {\'document_id\', \'sentence_id\'}.issubset(split.column_names)\n        digest = hashlib.sha256()\n        tokens_total = 0\n        previous_doc, previous_sentence = None, None\n        document_count = 0\n        for index, row in enumerate(split):\n            where = f\'{split_name}[{index}]\'\n            if row[\'id\'] != str(index):\n                raise ValueError(f\'{where}: id/order changed (expected id={index}).\')\n            tokens = row[\'tokens\']\n            if not isinstance(tokens, list) or not tokens or any(not isinstance(t, str) or not t for t in tokens):\n                raise ValueError(f\'{where}: empty or invalid token list.\')\n            if \'-DOCSTART-\' in tokens:\n                raise ValueError(f\'{where}: unexpected DOCSTART sentinel.\')\n            decoded = {}\n            for field in TAG_FIELDS:\n                tags = row[field]\n                if not isinstance(tags, list) or len(tags) != len(tokens):\n                    raise ValueError(f\'{where}: token/{field} length mismatch.\')\n                if any(isinstance(t, bool) or not isinstance(t, Integral) or not 0 <= t < len(names[field]) for t in tags):\n                    raise ValueError(f\'{where}: invalid {field} index.\')\n                decoded[field] = [names[field][int(t)] for t in tags]\n            previous = \'O\'\n            for position, tag in enumerate(decoded[\'ner_tags\']):\n                if tag.startswith(\'I-\') and previous not in (\'B-\' + tag[2:], tag):\n                    raise ValueError(f\'{where} token {position}: illegal IOB2 transition {previous} -> {tag}.\')\n                previous = tag\n            if has_documents:\n                doc, sentence = row[\'document_id\'], row[\'sentence_id\']\n                if any(isinstance(v, bool) or not isinstance(v, Integral) for v in (doc, sentence)):\n                    raise ValueError(f\'{where}: invalid document/sentence id.\')\n                if previous_doc is None or doc != previous_doc:\n                    if sentence != 0 or (previous_doc is not None and doc != previous_doc + 1):\n                        raise ValueError(f\'{where}: document order is not contiguous.\')\n                    document_count += 1\n                elif sentence != previous_sentence + 1:\n                    raise ValueError(f\'{where}: sentence order is not contiguous.\')\n                previous_doc, previous_sentence = doc, sentence\n            record = {\'id\': row[\'id\'], \'tokens\': tokens, **decoded}\n            digest.update((json.dumps(record, ensure_ascii=False, separators=(\',\', \':\')) + \'\\n\').encode(\'utf-8\'))\n            tokens_total += len(tokens)\n        report[\'splits\'][split_name] = {\n            \'rows\': len(split), \'tokens\': tokens_total, \'ordered_core_sha256\': digest.hexdigest(),\n            \'id_order_verified\': True, \'document_order_verified\': has_documents,\n            \'documents\': document_count if has_documents else None,\n        }\n    return report\n\n\ndef save_and_verify(dataset, project_root, run_id, report):\n    """Keep Arrow data on the runtime; verify persistence without putting data in ZIPs."""\n    from datasets import load_from_disk\n\n    destination = Path(project_root) / \'data\' / f\'conll2003_{run_id}\'\n    if destination.exists():\n        raise FileExistsError(\'This run was already saved. Rerun initialization for a new run_id.\')\n    dataset.save_to_disk(str(destination))\n    restored = load_from_disk(str(destination))\n    check = audit_dataset(restored)\n    if check != report:\n        raise ValueError(\'Saved dataset differs after read-back.\')\n    # Includes auxiliary columns and feature metadata, in addition to ordered core hashes.\n    files = [{\'path\': p.relative_to(destination).as_posix(), \'size_bytes\': p.stat().st_size,\n              \'sha256\': file_sha256(p)} for p in sorted(destination.rglob(\'*\')) if p.is_file()]\n    return {\'path\': str(destination), \'read_back_verified\': True, \'files\': files}\n', 'step2_eda.py': '"""Descriptive NER statistics. No training, model selection or test-set scoring."""\nimport csv\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nSPLITS = (\'train\', \'validation\', \'test\')\nTYPES = (\'PER\', \'ORG\', \'LOC\', \'MISC\')\nPAPER = {\n    \'train\': dict(documents=946, paper_sentences=14987, sentences=14041, tokens=203621,\n                  LOC=7140, MISC=3438, ORG=6321, PER=6600),\n    \'validation\': dict(documents=216, paper_sentences=3466, sentences=3250, tokens=51362,\n                       LOC=1837, MISC=922, ORG=1341, PER=1842),\n    \'test\': dict(documents=231, paper_sentences=3684, sentences=3453, tokens=46435,\n                 LOC=1668, MISC=702, ORG=1661, PER=1617),\n}\nDEFINITIONS = {\n    \'entity\': \'Strict IOB2 span, counted once at B-. End index is exclusive.\',\n    \'surface\': \'Exact tuple of original entity tokens; case-sensitive, no punctuation or whitespace normalization.\',\n    \'unseen_primary\': \'Surface absent among ALL training gold entity types. Mention rate weights occurrences; unique rate weights distinct surfaces within each type.\',\n    \'unseen_typed\': \'Surface absent from training gold entities of the SAME type; reported separately.\',\n    \'casefold\': \'Supplemental case-insensitive comparison uses str.casefold() on each token; original data are unchanged.\',\n    \'capitalization\': \'First character of the original first token is uppercase (str.isupper). Entity denominator: all mentions of that type. Sentence-O denominator: sentences whose token at index 0 has O; no punctuation skipping.\',\n    \'transition\': \'Only adjacent tokens within a sentence; no transitions across sentences. Probability divides by outgoing transitions for that label. Zero denominator is null (N/A).\',\n    \'ambiguity\': \'Exact token forms or complete entity surfaces with at least two distinct entity types WITHIN a split. O is excluded from entity types; O occurrence counts are supplemental.\',\n    \'vocabulary\': \'Includes punctuation and digits. Directed source-to-target type coverage = intersection / target vocabulary size; token coverage weights target token occurrences. Jaccard = intersection / union.\',\n    \'top_entities\': \'Top 10 per split and type, by mention count descending then token tuple ascending; plots show training entities.\',\n    \'paper_sentences\': \'Compare HF sentences directly to 14041/3250/3453; also compare HF sentences + document count to paper sentence count. DOCSTART explanation is an inference, not a reconstruction of the original files.\',\n    \'test_use\': \'Test statistics are descriptive only and must not select features, hyperparameters, preprocessing or models. No prediction scores are computed.\',\n}\n\n\ndef ratio(numerator, denominator):\n    return numerator / denominator if denominator else None\n\n\ndef extract_entities(tokens, tags):\n    """Return (start, end, type, token_tuple), rejecting illegal IOB2."""\n    if len(tokens) != len(tags):\n        raise ValueError(\'Token/tag length mismatch.\')\n    spans = []\n    start = kind = None\n    for index, tag in enumerate([*tags, \'O\']):\n        if tag != \'O\' and (tag[:2] not in (\'B-\', \'I-\') or tag[2:] not in TYPES):\n            raise ValueError(f\'Unknown tag: {tag}\')\n        if tag.startswith(\'I-\'):\n            if start is None or kind != tag[2:]:\n                raise ValueError(\'Illegal IOB2 continuation.\')\n        else:\n            if start is not None:\n                spans.append((start, index, kind, tuple(tokens[start:index])))\n                start = kind = None\n            if tag.startswith(\'B-\'):\n                start, kind = index, tag[2:]\n    return spans\n\n\ndef _normal_surface(surface, mode):\n    return surface if mode == \'exact\' else tuple(token.casefold() for token in surface)\n\n\ndef analyze(dataset, labels):\n    """One deterministic traversal; keeps distributions/aggregates rather than full rows."""\n    tables = {name: [] for name in (\'summary\', \'labels\', \'sentence_lengths\', \'entity_lengths\',\n              \'top_entities\', \'transitions\', \'capitalization\', \'unseen\', \'vocabulary\',\n              \'ambiguous_tokens\', \'ambiguous_entities\', \'order_spotchecks\', \'paper_comparison\')}\n    entities_by_split, vocab_by_split = {}, {}\n    for split in SPLITS:\n        rows = dataset[split]\n        label_counts, lengths, transitions, vocab = Counter(), Counter(), Counter(), Counter()\n        entity_counts = {kind: Counter() for kind in TYPES}\n        entity_lengths = {kind: Counter() for kind in TYPES}\n        caps = Counter()\n        token_types, surface_types = defaultdict(Counter), defaultdict(Counter)\n        outside_tokens = Counter()\n        docs, spots = set(), set(range(min(3, len(rows))))\n        middle = len(rows) // 2\n        spots.update(range(max(0, middle - 1), min(len(rows), middle + 2)))\n        previous_doc = None\n        boundary_found = False\n        for index, row in enumerate(rows):\n            tokens = row[\'tokens\']\n            tags = [labels[tag] for tag in row[\'ner_tags\']]\n            spans = extract_entities(tokens, tags)\n            label_counts.update(tags)\n            lengths[len(tokens)] += 1\n            vocab.update(tokens)\n            transitions.update(zip(tags, tags[1:]))\n            docs.add(row[\'document_id\'])\n            if previous_doc is not None and row[\'document_id\'] != previous_doc and not boundary_found:\n                spots.update([index - 1, index])\n                boundary_found = True\n            previous_doc = row[\'document_id\']\n            if tags and tags[0] == \'O\':\n                caps[\'sentence_O_total\'] += 1\n                caps[\'sentence_O_upper\'] += int(tokens[0][:1].isupper())\n            for token, tag in zip(tokens, tags):\n                if tag == \'O\':\n                    outside_tokens[token] += 1\n                else:\n                    token_types[token][tag[2:]] += 1\n            for start, end, kind, surface in spans:\n                entity_counts[kind][surface] += 1\n                entity_lengths[kind][end - start] += 1\n                surface_types[surface][kind] += 1\n                caps[kind] += int(surface[0][:1].isupper())\n            if len(spans) != sum(tag.startswith(\'B-\') for tag in tags):\n                raise ValueError(\'Span/B-tag count mismatch.\')\n        total_tokens = sum(vocab.values())\n        counts = {kind: sum(entity_counts[kind].values()) for kind in TYPES}\n        if sum(label_counts.values()) != total_tokens or sum(transitions.values()) != total_tokens - len(rows):\n            raise ValueError(\'Count/transition conservation failed.\')\n        summary = dict(split=split, sentences=len(rows), tokens=total_tokens, documents=len(docs),\n                       entities=sum(counts.values()), **counts)\n        tables[\'summary\'].append(summary)\n        for metric in (\'sentences\', \'tokens\', \'documents\', *TYPES):\n            expected = PAPER[split][metric]\n            tables[\'paper_comparison\'].append(dict(split=split, metric=metric,\n                observed=summary[metric], reference=expected, difference=summary[metric] - expected))\n        adjusted = len(rows) + len(docs)\n        tables[\'paper_comparison\'].append(dict(split=split, metric=\'sentences_plus_documents\',\n            observed=adjusted, reference=PAPER[split][\'paper_sentences\'],\n            difference=adjusted - PAPER[split][\'paper_sentences\']))\n        tables[\'paper_comparison\'].append(dict(split=split, metric=\'raw_sentences_vs_paper\',\n            observed=len(rows), reference=PAPER[split][\'paper_sentences\'],\n            difference=len(rows) - PAPER[split][\'paper_sentences\'], expected_difference=-PAPER[split][\'documents\']))\n        for label in labels:\n            tables[\'labels\'].append(dict(split=split, label=label, count=label_counts[label],\n                                        proportion=ratio(label_counts[label], total_tokens)))\n            outgoing = sum(transitions[(label, other)] for other in labels)\n            for other in labels:\n                n = transitions[(label, other)]\n                tables[\'transitions\'].append(dict(split=split, previous=label, following=other,\n                                                  count=n, probability=ratio(n, outgoing)))\n        for length, count in sorted(lengths.items()):\n            tables[\'sentence_lengths\'].append(dict(split=split, length=length, count=count))\n        for kind in TYPES:\n            for length, count in sorted(entity_lengths[kind].items()):\n                tables[\'entity_lengths\'].append(dict(split=split, type=kind, length=length,\n                                                     count=count, proportion=ratio(count, counts[kind])))\n            for rank, (surface, count) in enumerate(sorted(entity_counts[kind].items(),\n                                                           key=lambda x: (-x[1], x[0]))[:10], 1):\n                tables[\'top_entities\'].append(dict(split=split, type=kind, rank=rank,\n                                                  surface=\' \'.join(surface), count=count))\n            tables[\'capitalization\'].append(dict(split=split, metric=kind, uppercase=caps[kind],\n                                                 total=counts[kind], proportion=ratio(caps[kind], counts[kind])))\n        tables[\'capitalization\'].append(dict(split=split, metric=\'sentence_initial_O\',\n            uppercase=caps[\'sentence_O_upper\'], total=caps[\'sentence_O_total\'],\n            proportion=ratio(caps[\'sentence_O_upper\'], caps[\'sentence_O_total\'])))\n        for table, forms in [(\'ambiguous_tokens\', token_types), (\'ambiguous_entities\', surface_types)]:\n            ambiguous = [(form, values) for form, values in forms.items() if len(values) > 1]\n            for form, values in sorted(ambiguous, key=lambda x: (-sum(x[1].values()), x[0])):\n                item = dict(split=split, surface=form if table == \'ambiguous_tokens\' else \' \'.join(form),\n                            types=\'|\'.join(sorted(values)), total=sum(values.values()),\n                            **{kind: values[kind] for kind in TYPES})\n                if table == \'ambiguous_tokens\':\n                    item[\'O_count\'] = outside_tokens[form]\n                tables[table].append(item)\n        for index in sorted(spots):\n            row = rows[index]\n            tables[\'order_spotchecks\'].append(dict(split=split, row=index, id=row[\'id\'],\n                document_id=row[\'document_id\'], sentence_id=row[\'sentence_id\'],\n                preview=\' \'.join(row[\'tokens\'][:24])))\n        entities_by_split[split], vocab_by_split[split] = entity_counts, vocab\n\n    for mode in (\'exact\', \'casefold\'):\n        train_by_type = {kind: {_normal_surface(s, mode) for s in entities_by_split[\'train\'][kind]}\n                         for kind in TYPES}\n        train_any = set().union(*train_by_type.values())\n        for split in (\'validation\', \'test\'):\n            for kind in TYPES:\n                target = Counter()\n                for surface, count in entities_by_split[split][kind].items():\n                    target[_normal_surface(surface, mode)] += count\n                unseen = {s for s in target if s not in train_any}\n                unseen_typed = {s for s in target if s not in train_by_type[kind]}\n                n, unique = sum(target.values()), len(target)\n                tables[\'unseen\'].append(dict(split=split, normalization=mode, type=kind,\n                    mentions=n, unique_surfaces=unique,\n                    unseen_mentions=sum(target[s] for s in unseen), unseen_unique=len(unseen),\n                    mention_rate=ratio(sum(target[s] for s in unseen), n), unique_rate=ratio(len(unseen), unique),\n                    unseen_typed_mentions=sum(target[s] for s in unseen_typed), unseen_typed_unique=len(unseen_typed),\n                    typed_mention_rate=ratio(sum(target[s] for s in unseen_typed), n),\n                    typed_unique_rate=ratio(len(unseen_typed), unique)))\n        vocabs = {}\n        for split in SPLITS:\n            values = Counter()\n            for token, count in vocab_by_split[split].items():\n                values[token if mode == \'exact\' else token.casefold()] += count\n            vocabs[split] = values\n        for source in SPLITS:\n            for target in SPLITS:\n                a, b = set(vocabs[source]), set(vocabs[target])\n                overlap = a & b\n                covered = sum(vocabs[target][t] for t in overlap)\n                tables[\'vocabulary\'].append(dict(normalization=mode, source=source, target=target,\n                    source_types=len(a), target_types=len(b), shared_types=len(overlap), union_types=len(a | b),\n                    type_coverage=ratio(len(overlap), len(b)), jaccard=ratio(len(overlap), len(a | b)),\n                    covered_target_tokens=covered, target_tokens=sum(vocabs[target].values()),\n                    token_coverage=ratio(covered, sum(vocabs[target].values()))))\n    for row in tables[\'paper_comparison\']:\n        row.setdefault(\'expected_difference\', 0)\n        row[\'matches_expected\'] = row[\'difference\'] == row[\'expected_difference\']\n    return {\'definitions\': DEFINITIONS, \'labels_from_metadata\': list(labels), \'tables\': tables,\n            \'paper_comparison_passed\': all(row[\'matches_expected\'] for row in tables[\'paper_comparison\'])}\n\n\ndef ensure_dataset(project_root, accepted, run_id):\n    """Reuse the accepted Step 1 files, or redownload exactly the same pinned content."""\n    import step1_data as data_tools\n    from datasets import load_from_disk\n\n    project_root = Path(project_root)\n    path = project_root / \'data\' / Path(accepted[\'saved_data\'][\'path\']).name\n    if path.is_dir():\n        for record in accepted[\'saved_data\'][\'files\']:\n            file = path / record[\'path\']\n            if not file.is_file() or data_tools.file_sha256(file) != record[\'sha256\']:\n                raise ValueError(\'Saved Step 1 data changed. Stop and report this error.\')\n        dataset = load_from_disk(str(path))\n        source = accepted[\'source\']\n        recovery = \'reused-verified-step1-files\'\n    else:\n        dataset, source = data_tools.load_pinned_dataset(project_root)\n        if source[\'source_files\'] != accepted[\'source\'][\'source_files\']:\n            raise ValueError(\'Pinned downloads differ from the accepted Step 1 source hashes.\')\n        recovery = \'redownloaded-pinned-step1-source\'\n    audit = data_tools.audit_dataset(dataset)\n    if audit != accepted[\'audit\']:\n        raise ValueError(\'Dataset differs from the accepted Step 1 ordered audit.\')\n    if recovery.startswith(\'redownloaded\'):\n        saved = data_tools.save_and_verify(dataset, project_root, run_id, audit)\n    else:\n        saved = {**accepted[\'saved_data\'], \'path\': str(path)}\n    record = dict(source=source, saved_data=saved, audit=audit, recovery=recovery, run_id=run_id)\n    data_tools.write_json(project_root / \'config\' / \'dataset_latest.json\', record)\n    return dataset, record\n\n\ndef write_tables(report, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=False)\n    for name, rows in report[\'tables\'].items():\n        path = destination / f\'{name}.csv\'\n        with path.open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as stream:\n            fields = list(rows[0]) if rows else [\'split\', \'surface\', \'types\', \'total\', *TYPES]\n            writer = csv.DictWriter(stream, fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    (destination / \'eda_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    lines = [\'# Step 2 descriptive EDA\', \'\', \'No model training or prediction scoring.\', \'\',\n             \'Reference: https://aclanthology.org/W03-0419.pdf (Tables 1 and 2).\', \'\', \'## Definitions\', \'\']\n    lines.extend(f\'- {key}: {value}\' for key, value in report[\'definitions\'].items())\n    lines += [\'\', \'## Observed counts\', \'\', \'| Split | Sentences | Tokens | Entities | PER | ORG | LOC | MISC |\',\n              \'|---|---:|---:|---:|---:|---:|---:|---:|\']\n    for row in report[\'tables\'][\'summary\']:\n        lines.append(\'| \' + \' | \'.join(str(row[k]) for k in (\'split\', \'sentences\', \'tokens\', \'entities\', *TYPES)) + \' |\')\n    lines += [\'\', f"Paper count comparison passed: {report[\'paper_comparison_passed\']}.",\n              \'Raw paper sentence counts include an additional document-sized difference; see paper_comparison.csv.\',\n              \'Use train-only observations when considering features. Test statistics are descriptive, not a selection criterion.\',\n              \'PNG and vector PDF figures are stored in the corresponding figures/step2_<run_id> directory.\']\n    (destination / \'EDA_README.md\').write_text(\'\\n\'.join(lines) + \'\\n\', encoding=\'utf-8\')\n\n\nFIGURE_NAMES = (\'01_label_distribution\', \'02_sentence_lengths\', \'03_entity_lengths\',\n                \'04_top_entities_train\', \'05_tag_transitions\', \'06_capitalization\',\n                \'07_unseen_entities\', \'08_vocabulary_overlap\', \'09_ambiguity_train\')\n\n\ndef plot_report(report, destination):\n    """Nine report figures, each exported as a PNG and a vector PDF."""\n    import matplotlib.pyplot as plt\n    import numpy as np\n    from matplotlib.ticker import PercentFormatter, MaxNLocator\n    from textwrap import fill\n\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    tables, labels = report[\'tables\'], report[\'labels_from_metadata\']\n    colors = [\'#2670AA\', \'#D17B22\', \'#36846B\', \'#A75085\']\n    paths = []\n    style = {\'font.family\': \'DejaVu Sans\', \'font.size\': 10, \'axes.spines.top\': False,\n             \'axes.spines.right\': False, \'savefig.facecolor\': \'white\', \'figure.facecolor\': \'white\'}\n\n    def selected(name, **where):\n        return [r for r in tables[name] if all(r[k] == v for k, v in where.items())]\n\n    def save(fig, name):\n        fig.savefig(destination / (name + \'.png\'), dpi=170, bbox_inches=\'tight\')\n        fig.savefig(destination / (name + \'.pdf\'), bbox_inches=\'tight\')\n        paths.append(str(destination / (name + \'.png\')))\n        plt.close(fig)\n\n    def number(value):\n        return float(\'nan\') if value is None else value\n\n    with plt.rc_context(style):\n        fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), layout=\'constrained\', sharey=True)\n        for ax, split, color in zip(axes, SPLITS, colors):\n            rows = selected(\'labels\', split=split)\n            ax.bar([r[\'label\'] for r in rows], [r[\'proportion\'] for r in rows], color=color)\n            ax.set(title=split, xlabel=\'Token label (includes O)\')\n            ax.tick_params(axis=\'x\', rotation=55)\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n        axes[0].set_ylabel(\'Share of tokens\')\n        fig.suptitle(\'NER label distribution\')\n        save(fig, FIGURE_NAMES[0])\n\n        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout=\'constrained\')\n        maximum = max(r[\'length\'] for r in tables[\'sentence_lengths\'])\n        bins = np.arange(0.5, maximum + 5.5, 5)\n        for ax, split, color in zip(axes, SPLITS, colors):\n            rows = selected(\'sentence_lengths\', split=split)\n            ax.hist([r[\'length\'] for r in rows], bins=bins, weights=[r[\'count\'] for r in rows], color=color)\n            ax.set(title=f\'{split} (max {max(r["length"] for r in rows)})\', xlabel=\'Tokens per sentence\', ylabel=\'Sentences\', xlim=(0, maximum + 1))\n        fig.suptitle(\'Sentence lengths — all sentences, no tail truncation\')\n        save(fig, FIGURE_NAMES[1])\n\n        fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout=\'constrained\')\n        for ax, kind in zip(axes.flat, TYPES):\n            kind_rows = selected(\'entity_lengths\', type=kind)\n            maximum = max((r[\'length\'] for r in kind_rows), default=1)\n            x = np.arange(1, maximum + 1)\n            for split, color in zip(SPLITS, colors):\n                rows = selected(\'entity_lengths\', split=split, type=kind)\n                counts = {r[\'length\']: r[\'count\'] for r in rows}\n                total = sum(counts.values())\n                probabilities = [counts.get(length, 0) / total if total else np.nan for length in x]\n                ax.plot(x, probabilities, marker=\'o\', ms=3, label=split, color=color)\n            ax.set(title=kind, xlabel=\'Tokens per entity\', ylabel=\'Share within split/type\',\n                   ylim=(0, 1.05), xlim=(.5, maximum + .5))\n            ax.xaxis.set_major_locator(MaxNLocator(integer=True))\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n            ax.legend(fontsize=8)\n        fig.suptitle(\'Entity length distributions\')\n        save(fig, FIGURE_NAMES[2])\n\n        fig, axes = plt.subplots(2, 2, figsize=(16, 12), layout=\'constrained\')\n        for ax, kind, color in zip(axes.flat, TYPES, colors):\n            rows = selected(\'top_entities\', split=\'train\', type=kind)\n            ax.barh(range(len(rows)), [r[\'count\'] for r in rows], color=color)\n            ax.set_yticks(range(len(rows)), [fill(r[\'surface\'], 26) for r in rows], fontsize=9)\n            ax.invert_yaxis()\n            ax.set(title=kind, xlabel=\'Entity mentions in training data\')\n        fig.suptitle(\'Ten most frequent entity surfaces per type (train only)\')\n        save(fig, FIGURE_NAMES[3])\n\n        fig, axes = plt.subplots(1, 3, figsize=(20, 7), layout=\'constrained\')\n        for ax, split in zip(axes, SPLITS):\n            lookup = {(r[\'previous\'], r[\'following\']): r[\'probability\'] for r in selected(\'transitions\', split=split)}\n            matrix = np.array([[number(lookup[(a, b)]) for b in labels] for a in labels])\n            im = ax.imshow(np.ma.masked_invalid(matrix), vmin=0, vmax=1, cmap=\'Blues\')\n            ax.set_xticks(range(len(labels)), labels, rotation=60, ha=\'right\')\n            ax.set_yticks(range(len(labels)), labels)\n            ax.set(title=split, xlabel=\'Next label\', ylabel=\'Current label\')\n            for i in range(len(labels)):\n                for j in range(len(labels)):\n                    v = matrix[i, j]\n                    ax.text(j, i, \'N/A\' if np.isnan(v) else f\'{100*v:.0f}\', ha=\'center\', va=\'center\',\n                            fontsize=7, color=\'white\' if v > .55 else \'#172B40\')\n        fig.colorbar(im, ax=axes, shrink=.65, label=\'Row probability\')\n        fig.suptitle(\'Within-sentence label transitions — cell values in percent\')\n        save(fig, FIGURE_NAMES[4])\n\n        fig, ax = plt.subplots(figsize=(12, 5.5), layout=\'constrained\')\n        categories = [*TYPES, \'sentence_initial_O\']\n        x = np.arange(len(categories))\n        for i, (split, color) in enumerate(zip(SPLITS, colors)):\n            lookup = {r[\'metric\']: r for r in selected(\'capitalization\', split=split)}\n            ax.bar(x + (i-1)*.25, [number(lookup[k][\'proportion\']) for k in categories], width=.24, color=color, label=split)\n        ax.set_xticks(x, [*TYPES, \'Sentence-first O token\'])\n        ax.set(ylabel=\'Uppercase first character / eligible occurrences\', ylim=(0, 1.05),\n               title=\'Capitalization of entity starts and sentence-initial non-entities\')\n        ax.yaxis.set_major_formatter(PercentFormatter(1))\n        ax.legend()\n        save(fig, FIGURE_NAMES[5])\n\n        fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout=\'constrained\', sharey=True)\n        x = np.arange(len(TYPES))\n        for ax, rate, title in zip(axes, (\'mention_rate\', \'unique_rate\'), (\'By mention occurrence\', \'By unique surface\')):\n            for i, (split, color) in enumerate(zip((\'validation\', \'test\'), colors[1:])):\n                lookup = {r[\'type\']: r for r in selected(\'unseen\', split=split, normalization=\'exact\')}\n                ax.bar(x + (i-.5)*.34, [number(lookup[k][rate]) for k in TYPES], width=.33, label=split, color=color)\n            ax.set_xticks(x, TYPES)\n            ax.set(title=title, ylabel=\'Unseen proportion\', ylim=(0, 1.05))\n            ax.yaxis.set_major_formatter(PercentFormatter(1))\n            ax.legend()\n        fig.suptitle(\'Entity surfaces unseen among training entities (exact case, any type)\')\n        save(fig, FIGURE_NAMES[6])\n\n        fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), layout=\'constrained\')\n        for ax, mode in zip(axes, (\'exact\', \'casefold\')):\n            lookup = {(r[\'source\'], r[\'target\']): r[\'type_coverage\'] for r in selected(\'vocabulary\', normalization=mode)}\n            matrix = np.array([[number(lookup[(a, b)]) for b in SPLITS] for a in SPLITS])\n            im = ax.imshow(matrix, cmap=\'YlGnBu\', vmin=0, vmax=1)\n            ax.set_xticks(range(3), SPLITS)\n            ax.set_yticks(range(3), SPLITS)\n            ax.set(title=mode, xlabel=\'Target vocabulary (denominator)\', ylabel=\'Source vocabulary\')\n            for i in range(3):\n                for j in range(3):\n                    ax.text(j, i, f\'{matrix[i,j]:.1%}\', ha=\'center\', va=\'center\', color=\'white\' if matrix[i,j] > .65 else \'black\')\n        fig.colorbar(im, ax=axes, shrink=.75, label=\'Target type coverage\')\n        fig.suptitle(\'Directed vocabulary overlap (includes punctuation and digits)\')\n        save(fig, FIGURE_NAMES[7])\n\n        fig, axes = plt.subplots(1, 2, figsize=(16, 10), layout=\'constrained\')\n        for ax, name, title in zip(axes, (\'ambiguous_tokens\', \'ambiguous_entities\'), (\'Token forms\', \'Complete entity surfaces\')):\n            rows = selected(name, split=\'train\')[:15]\n            if not rows:\n                ax.text(.5, .5, \'No multi-type forms\', ha=\'center\', va=\'center\', transform=ax.transAxes)\n                ax.set_title(title)\n                continue\n            left = np.zeros(len(rows))\n            for kind, color in zip(TYPES, colors):\n                values = np.array([r[kind] for r in rows])\n                ax.barh(range(len(rows)), values, left=left, label=kind, color=color)\n                left += values\n            ax.set_yticks(range(len(rows)), [fill(r[\'surface\'], 25) for r in rows], fontsize=9)\n            ax.invert_yaxis()\n            ax.set(title=title, xlabel=\'Entity-token occurrences\' if name == \'ambiguous_tokens\' else \'Entity mentions\')\n            ax.legend(fontsize=8)\n        fig.suptitle(\'Multi-type ambiguity in training data (up to 15 most frequent forms; O excluded)\')\n        save(fig, FIGURE_NAMES[8])\n    return paths\n', 'step3_alignment.py': '"""Lossless word/subword alignment, whole-word chunks and prediction reconstruction."""\nimport hashlib\nimport json\nfrom collections import Counter\nfrom numbers import Integral\nfrom pathlib import Path\n\nIGNORE_INDEX = -100\nMAX_LENGTH = 256  # Includes special tokens; fixed before observing tokenized test lengths.\nTOKENIZER_SPECS = {\n    \'bert_base_cased\': {\n        \'repo_id\': \'google-bert/bert-base-cased\',\n        \'revision\': \'cd5ef92a9fb2f889e972770a36d4ed042daf221e\',\n        \'files\': [\'config.json\', \'tokenizer_config.json\', \'tokenizer.json\', \'vocab.txt\'],\n        \'kwargs\': {},\n    },\n    \'roberta_base\': {\n        \'repo_id\': \'FacebookAI/roberta-base\',\n        \'revision\': \'e2da8e2f811d1448a5b465c236feacd80ffbac7b\',\n        \'files\': [\'config.json\', \'tokenizer_config.json\', \'tokenizer.json\', \'vocab.json\', \'merges.txt\'],\n        \'kwargs\': {\'add_prefix_space\': True},\n    },\n}\nPOLICY = {\n    \'max_length\': MAX_LENGTH,\n    \'includes_special_tokens\': True,\n    \'scheme\': \'IOB2 labels retained from metadata; no relabeling at chunk boundaries.\',\n    \'supervision\': \'First subword of each original word only; continuation/special/padding labels are -100.\',\n    \'chunking\': \'Greedy nonoverlapping complete-word chunks based only on tokenized word lengths; never gold entities.\',\n    \'oversized_word\': \'Raise an error if one word exceeds the content budget or produces no subword; never silently discard it.\',\n    \'evaluation\': \'Reassemble each original sentence in word order before entity metrics; do not score chunks separately.\',\n    \'context_limit\': \'Whole-word chunking can cut through an entity and limits cross-chunk context; original gold labels are preserved.\',\n    \'padding\': \'Dynamic right-padding per batch; strip alignment metadata before passing a batch to a model.\',\n    \'model_work\': \'Tokenizer preparation only: no model weights, training, predictions or F1 scoring.\',\n}\n\n\ndef load_tokenizer(key, project_root):\n    from huggingface_hub import snapshot_download\n    from transformers import AutoTokenizer\n    from step1_data import file_sha256\n\n    spec = TOKENIZER_SPECS[key]\n    snapshot = Path(snapshot_download(repo_id=spec[\'repo_id\'], revision=spec[\'revision\'],\n        allow_patterns=spec[\'files\'], cache_dir=str(Path(project_root) / \'data\' / \'tokenizer_cache\'), token=False))\n    files = []\n    for filename in spec[\'files\']:\n        path = snapshot / filename\n        if not path.is_file():\n            raise FileNotFoundError(f\'Missing pinned tokenizer file: {filename}\')\n        files.append(dict(path=filename, size_bytes=path.stat().st_size, sha256=file_sha256(path)))\n    tokenizer = AutoTokenizer.from_pretrained(str(snapshot), use_fast=True, local_files_only=True,\n                                               trust_remote_code=False, **spec[\'kwargs\'])\n    tokenizer.padding_side = \'right\'\n    if not tokenizer.is_fast or tokenizer.pad_token_id is None:\n        raise ValueError(\'A fast tokenizer with a padding token is required.\')\n    if MAX_LENGTH > tokenizer.model_max_length:\n        raise ValueError(\'Chosen maximum exceeds the tokenizer model limit.\')\n    record = dict(key=key, **spec, source_files=files, tokenizer_class=type(tokenizer).__name__,\n                  model_max_length=tokenizer.model_max_length, padding_side=tokenizer.padding_side,\n                  special_tokens_per_sentence=tokenizer.num_special_tokens_to_add(pair=False))\n    return tokenizer, record\n\n\ndef whole_word_ranges(piece_counts, budget):\n    """Return contiguous [start,end) word ranges, with no overlap or truncation."""\n    if budget < 1 or not piece_counts:\n        raise ValueError(\'Positive content budget and at least one word are required.\')\n    ranges, start, used = [], 0, 0\n    for index, count in enumerate(piece_counts):\n        if count < 1 or count > budget:\n            raise ValueError(f\'Word {index} has {count} pieces, outside content budget 1..{budget}.\')\n        if used + count > budget:\n            ranges.append((start, index))\n            start, used = index, 0\n        used += count\n    ranges.append((start, len(piece_counts)))\n    return ranges\n\n\ndef encode_sentence(row, sentence_index, tokenizer, label_names, max_length=MAX_LENGTH):\n    words, gold = row[\'tokens\'], row[\'ner_tags\']\n    if not words or len(words) != len(gold):\n        raise ValueError(\'Missing words or token/gold length mismatch.\')\n    if any(isinstance(v, bool) or not isinstance(v, Integral) or not 0 <= v < len(label_names) for v in gold):\n        raise ValueError(\'Invalid gold label index.\')\n    special_count = tokenizer.num_special_tokens_to_add(pair=False)\n    if max_length > tokenizer.model_max_length:\n        raise ValueError(\'Chosen max_length exceeds the model limit.\')\n    flat = tokenizer(words, is_split_into_words=True, add_special_tokens=False, truncation=False,\n                     padding=False, return_attention_mask=False, return_token_type_ids=False, verbose=False)\n    flat_ids, word_ids = flat[\'input_ids\'], flat.word_ids()\n    if len(flat_ids) != len(word_ids) or any(w is None for w in word_ids):\n        raise ValueError(\'Unexpected special/unmapped token in plain word encoding.\')\n    runs = [w for i, w in enumerate(word_ids) if i == 0 or word_ids[i-1] != w]\n    if runs != list(range(len(words))):\n        raise ValueError(\'Tokenizer omitted, reordered or fragmented a word mapping.\')\n    counts = Counter(word_ids)\n    pieces = [counts[i] for i in range(len(words))]\n    boundaries = [0]\n    for count in pieces:\n        boundaries.append(boundaries[-1] + count)\n    ranges = whole_word_ranges(pieces, max_length - special_count)\n    chunks = []\n    for chunk_index, (start, end) in enumerate(ranges):\n        encoded = tokenizer(words[start:end], is_split_into_words=True, add_special_tokens=True,\n                            truncation=False, padding=False, return_special_tokens_mask=True)\n        local_ids = encoded.word_ids()\n        if [w for i,w in enumerate(local_ids) if w is not None and (i == 0 or local_ids[i-1] != w)] != list(range(end-start)):\n            raise ValueError(\'Chunk word mapping changed.\')\n        content = [token for token,w in zip(encoded[\'input_ids\'], local_ids) if w is not None]\n        if content != flat_ids[boundaries[start]:boundaries[end]]:\n            raise ValueError(\'Chunk tokenization differs from full-sentence tokenization.\')\n        if len(encoded[\'input_ids\']) > max_length:\n            raise ValueError(\'Whole-word chunk still exceeds max_length.\')\n        labels, mapped, previous = [], [], None\n        for w, special in zip(local_ids, encoded[\'special_tokens_mask\']):\n            if (w is None) != bool(special):\n                raise ValueError(\'Special-token mask disagrees with word mapping.\')\n            mapped.append(-1 if w is None else start + w)\n            labels.append(int(gold[start+w]) if w is not None and w != previous else IGNORE_INDEX)\n            previous = w\n        chunk = {k:list(encoded[k]) for k in tokenizer.model_input_names if k in encoded}\n        chunk.update(labels=labels, word_ids=mapped, sentence_index=sentence_index,\n                     sentence_id=str(row[\'id\']), chunk_index=chunk_index, word_start=start, word_end=end)\n        chunks.append(chunk)\n    stats = dict(sentence_index=sentence_index, sentence_id=str(row[\'id\']), words=len(words),\n                 full_length=len(flat_ids)+special_count, chunks=len(chunks),\n                 multi_piece_words=sum(n > 1 for n in pieces),\n                 unknown_pieces=sum(v == tokenizer.unk_token_id for v in flat_ids))\n    return chunks, stats\n\n\ndef reassemble_word_predictions(chunks, prediction_ids, word_counts, num_labels):\n    """Class ids -> original words. Ignores special/continuation/padding predictions.\n\n    Alignment uses word_ids, not gold labels. Prediction rows must follow chunk rows.\n    This also validates a gold-label round trip without computing a model metric.\n    """\n    if len(chunks) != len(prediction_ids):\n        raise ValueError(\'Prediction/chunk count mismatch.\')\n    restored = [[None] * n for n in word_counts]\n    for chunk, predictions in zip(chunks, prediction_ids):\n        if len(predictions) < len(chunk[\'input_ids\']):\n            raise ValueError(\'Prediction row shorter than its encoded chunk.\')\n        sentence = chunk[\'sentence_index\']\n        if not 0 <= sentence < len(restored):\n            raise ValueError(\'Unknown sentence index.\')\n        previous = -1\n        selected = []\n        for index, word in enumerate(chunk[\'word_ids\']):\n            if word >= 0 and word != previous:\n                if not 0 <= word < len(restored[sentence]):\n                    raise ValueError(\'Word index outside original sentence.\')\n                value = predictions[index]\n                if isinstance(value, bool) or not isinstance(value, Integral) or not 0 <= value < num_labels:\n                    raise ValueError(\'Invalid prediction at a first-subword position.\')\n                if restored[sentence][word] is not None:\n                    raise ValueError(\'Duplicate prediction for an original word.\')\n                restored[sentence][word] = int(value)\n                selected.append(word)\n            previous = word\n        if selected != list(range(chunk[\'word_start\'], chunk[\'word_end\'])):\n            raise ValueError(\'Chunk word range differs from first-subword mapping.\')\n    if any(value is None for row in restored for value in row):\n        raise ValueError(\'At least one original word has no prediction.\')\n    return restored\n\n\ndef model_features(chunk, tokenizer):\n    """Keep metadata out of the model/collator input, retaining it in the Arrow sidecar."""\n    return {k:chunk[k] for k in [*tokenizer.model_input_names, \'labels\'] if k in chunk}\n\n\ndef validate_chunks(chunks, source_rows, num_labels, max_length=MAX_LENGTH):\n    counts = [len(row[\'tokens\']) for row in source_rows]\n    labels = [row[\'ner_tags\'] for row in source_rows]\n    special = continuation = supervised = 0\n    lengths = []\n    for chunk in chunks:\n        n = len(chunk[\'input_ids\'])\n        if n > max_length or n < 1 or any(len(chunk[k]) != n for k in (\'attention_mask\',\'labels\',\'word_ids\')):\n            raise ValueError(\'Chunk lengths are inconsistent.\')\n        if \'token_type_ids\' in chunk and len(chunk[\'token_type_ids\']) != n:\n            raise ValueError(\'Token-type ids have an inconsistent length.\')\n        if any(isinstance(w, bool) or not isinstance(w, Integral) or w < -1 for w in chunk[\'word_ids\']):\n            raise ValueError(\'Invalid stored word index.\')\n        if any(v != 1 for v in chunk[\'attention_mask\']):\n            raise ValueError(\'Saved chunks must be unpadded.\')\n        previous = -1\n        for word, value in zip(chunk[\'word_ids\'], chunk[\'labels\']):\n            first = word >= 0 and word != previous\n            if not first:\n                if value != IGNORE_INDEX:\n                    raise ValueError(\'Special or continuation token carries supervision.\')\n                special += int(word < 0)\n                continuation += int(word >= 0)\n            else:\n                supervised += 1\n            previous = word\n        lengths.append(n)\n    restored = reassemble_word_predictions(chunks, [r[\'labels\'] for r in chunks], counts, num_labels)\n    if restored != labels or supervised != sum(counts):\n        raise ValueError(\'Gold labels did not round-trip for every original word exactly once.\')\n    return dict(original_words=sum(counts), supervised_words=supervised, dropped_words=0,\n                duplicated_words=0, masked_special_tokens=special, masked_continuation_tokens=continuation,\n                max_chunk_length=max(lengths, default=0), gold_roundtrip_passed=True)\n\n\ndef padding_check(tokenizer, chunks):\n    from transformers import DataCollatorForTokenClassification\n    import numpy as np\n\n    chosen = [min(chunks, key=lambda x:len(x[\'input_ids\'])), max(chunks, key=lambda x:len(x[\'input_ids\']))]\n    collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True,\n                                                  label_pad_token_id=IGNORE_INDEX, return_tensors=\'np\')\n    batch = collator([model_features(row, tokenizer) for row in chosen])\n    for i, row in enumerate(chosen):\n        n = len(row[\'input_ids\'])\n        if list(batch[\'labels\'][i,:n]) != row[\'labels\']:\n            raise ValueError(\'Collator changed original labels.\')\n    masked = batch[\'attention_mask\'] == 0\n    if not np.all(batch[\'labels\'][masked] == IGNORE_INDEX):\n        raise ValueError(\'Padding labels must be -100.\')\n    return dict(batch_shape=list(batch[\'input_ids\'].shape), padding_positions=int(masked.sum()), padding_labels_verified=True)\n\n\ndef records_sha256(records):\n    digest = hashlib.sha256()\n    for row in records:\n        digest.update((json.dumps(row,sort_keys=True,separators=(\',\',\':\'))+\'\\n\').encode(\'utf-8\'))\n    return digest.hexdigest()\n\n\ndef synthetic_alignment_check(tokenizer, label_names):\n    """Exercise multi-piece words, forced long-entity splitting and dynamic padding."""\n    ids = {label: i for i, label in enumerate(label_names)}\n    rows = [\n        dict(id=\'synthetic-short\', tokens=[\'Jane\', \'Villanueva\', \'visited\', \'Hong\', \'Kong\', \'.\'],\n             ner_tags=[ids[t] for t in [\'B-PER\', \'I-PER\', \'O\', \'B-LOC\', \'I-LOC\', \'O\']]),\n        dict(id=\'synthetic-long\', tokens=[\'Jane\'] + [\'Villanueva\']*180 + [\'.\'],\n             ner_tags=[ids[\'B-PER\']] + [ids[\'I-PER\']]*180 + [ids[\'O\']]),\n        dict(id=\'synthetic-symbols\', tokens=[\'U.N.\', \'München\', \'中文\', \'[CLS]\', \'<s>\', \'1996-08-22\'],\n             ner_tags=[ids[\'O\']]*6),\n    ]\n    chunks, lengths = [], []\n    for index, row in enumerate(rows):\n        encoded, info = encode_sentence(row, index, tokenizer, label_names, max_length=64)\n        chunks.extend(encoded); lengths.append(info)\n    audit = validate_chunks(chunks, rows, len(label_names), max_length=64)\n    padded = padding_check(tokenizer, chunks)\n    if lengths[1][\'chunks\'] <= 1 or audit[\'masked_continuation_tokens\'] == 0 or padded[\'padding_positions\'] == 0:\n        raise ValueError(\'Synthetic test did not exercise splitting, continuation and padding.\')\n    restored = reassemble_word_predictions(chunks, [r[\'labels\'] + [0]*3 for r in chunks],\n                                          [len(r[\'tokens\']) for r in rows], len(label_names))\n    if restored != [r[\'ner_tags\'] for r in rows]:\n        raise ValueError(\'Synthetic reconstruction with padded predictions failed.\')\n    return dict(synthetic_only=True, forced_max_length=64, sentences=lengths, audit=audit, padding=padded)\n\n\ndef prepare_tokenizer_dataset(dataset, tokenizer, label_names, destination, key):\n    from datasets import Dataset, DatasetDict, load_from_disk\n    from step1_data import file_sha256\n\n    destination = Path(destination)\n    if destination.exists():\n        raise FileExistsError(\'Output exists. Reinitialize Step 3 to create a fresh run_id.\')\n    prepared, summaries, histograms, overflow, examples = {}, [], [], [], []\n    expected_hashes = {}\n    for split, source in dataset.items():\n        records, lengths = [], Counter()\n        multi = unknown = long_count = 0\n        for index, row in enumerate(source):\n            chunks, info = encode_sentence(row, index, tokenizer, label_names)\n            records.extend(chunks)\n            lengths[info[\'full_length\']] += 1\n            multi += info[\'multi_piece_words\']; unknown += info[\'unknown_pieces\']\n            if info[\'full_length\'] > MAX_LENGTH:\n                long_count += 1\n                overflow.append(dict(tokenizer=key, split=split, **info))\n            if split == \'train\' and (index < 2 or (info[\'chunks\'] > 1 and not any(e[\'example_kind\']==\'long_train\' for e in examples))):\n                for chunk in chunks:\n                    for pos, (piece, word, label) in enumerate(zip(tokenizer.convert_ids_to_tokens(chunk[\'input_ids\']),chunk[\'word_ids\'],chunk[\'labels\'])):\n                        examples.append(dict(tokenizer=key, example_kind=\'first_train\' if index<2 else \'long_train\',\n                            sentence_index=index, chunk_index=chunk[\'chunk_index\'], position=pos, subword=piece,\n                            word_index=word, original_word=\'\' if word<0 else row[\'tokens\'][word],\n                            label_id=label, label=\'IGNORED\' if label==IGNORE_INDEX else label_names[label]))\n            if (index+1) % 5000 == 0:\n                print(f\'{key}/{split}: aligned {index+1}/{len(source)} sentences\')\n        validation = validate_chunks(records, source, len(label_names))\n        padded = padding_check(tokenizer, records)\n        expected_hashes[split] = records_sha256(records)\n        prepared[split] = Dataset.from_list(records)\n        summaries.append(dict(tokenizer=key,split=split,sentences=len(source),chunks=len(records),\n            over_max_length_sentences=long_count,max_full_length=max(lengths),\n            multi_piece_words=multi,unknown_pieces=unknown,**validation,**padded,\n            ordered_alignment_sha256=expected_hashes[split]))\n        histograms.extend(dict(tokenizer=key,split=split,full_length=n,sentences=count) for n,count in sorted(lengths.items()))\n    result = DatasetDict(prepared)\n    result.save_to_disk(str(destination / \'dataset\'))\n    tokenizer.save_pretrained(str(destination / \'tokenizer\'))\n    reloaded = load_from_disk(str(destination / \'dataset\'))\n    for split, source in dataset.items():\n        if records_sha256(reloaded[split]) != expected_hashes[split]:\n            raise ValueError(\'Alignment files changed after save/read-back.\')\n        validate_chunks(reloaded[split], source, len(label_names))\n    files = [dict(path=p.relative_to(destination).as_posix(),size_bytes=p.stat().st_size,sha256=file_sha256(p))\n             for p in sorted(destination.rglob(\'*\')) if p.is_file()]\n    return dict(key=key, path=str(destination), summaries=summaries, length_histograms=histograms,\n                overflow=overflow, alignment_examples=examples, saved_files=files, read_back_verified=True)\n', 'step5a_memory.py': '"""Train-only phrase memory baseline; deterministic decoding and reusable evaluation."""\nimport hashlib\nimport json\nfrom collections import Counter, defaultdict\nfrom numbers import Integral\nfrom pathlib import Path\n\nimport metrics\nimport prediction_io\nfrom experiment_protocol import canonical_bytes, fingerprint\n\nVERSION = \'ner-memory-baseline-v1\'\nPOLICY = {\n    \'version\': VERSION,\n    \'fit\': \'Official train only; memorize complete gold entity token tuples, never subphrases.\',\n    \'unique\': \'Keep a surface only when its complete training entities have exactly one NE type; discard all multi-type surfaces regardless of frequency. O is not an entity type.\',\n    \'matching\': \'Exact original tokens, case-sensitive; no normalization, substring matches or external gazetteers.\',\n    \'overlap\': \'Collect all sentence-local matches; accept by descending token length, then ascending start offset, skipping any occupied token. Finally sort by start.\',\n    \'tie_break\': \'Equal-length overlapping matches prefer the earlier start. A retained surface has only one type.\',\n    \'output\': \'IOB2: B at each accepted entity start, I inside, O elsewhere. Adjacent entities remain separate. Never cross sentence boundaries.\',\n    \'selection\': \'One preregistered configuration; no hyperparameters, CV, frequency cutoff or validation/test-based changes.\',\n    \'interpretation\': \'Implements the paper description. Case sensitivity and equal-length/crossing-overlap resolution are explicit project choices; exact identity to the historical executable is not established.\',\n    \'reference\': \'https://aclanthology.org/W03-0419.pdf (section 3.4, Table 5)\',\n}\nPAPER = {\'precision\': 0.7191, \'recall\': 0.5090, \'f1\': 0.5961}\nBOOTSTRAP = {\'n_resamples\': 2000, \'confidence\': 0.95, \'seed\': 2026}\n\n\ndef _tokens(tokens):\n    if not isinstance(tokens, (list, tuple)) or any(not isinstance(t, str) or not t for t in tokens):\n        raise ValueError(\'Expected original nonempty token strings, not raw text or labelled rows.\')\n    return list(tokens)\n\n\ndef _labels(label_names):\n    if len(label_names) != len(metrics.LABELS) or set(label_names) != metrics.LABELS:\n        raise ValueError(\'Use the complete metadata-derived label order.\')\n\n\nclass MemoryBaseline:\n    """Predict takes tokens only. No gold, POS, chunk labels or split context."""\n\n    def __init__(self, payload):\n        if payload.get(\'version\') != VERSION or payload.get(\'policy\') != POLICY:\n            raise ValueError(\'Memory model version/policy mismatch.\')\n        _labels(payload[\'label_names\'])\n        self.payload = json.loads(json.dumps(payload))\n        self._trie = {}\n        seen = set()\n        for item in payload[\'entries\']:\n            tokens = tuple(_tokens(item[\'tokens\']))\n            if not tokens or tokens in seen or item[\'type\'] not in metrics.TYPES or item[\'mentions\'] < 1:\n                raise ValueError(\'Invalid or duplicate dictionary entry.\')\n            seen.add(tokens)\n            node = self._trie\n            for token in tokens:\n                node = node.setdefault(token, {})\n            node[None] = item[\'type\']\n\n    @classmethod\n    def fit(cls, train_rows, label_names):\n        _labels(label_names)\n        counts = defaultdict(Counter)\n        digest = hashlib.sha256()\n        n_sentences = n_words = 0\n        ids = set()\n        for row in train_rows:\n            tokens = _tokens(row[\'tokens\'])\n            identity = str(row[\'id\'])\n            if not tokens or identity in ids or not identity or len(tokens) != len(row[\'ner_tags\']):\n                raise ValueError(\'Invalid training row/id/length.\')\n            ids.add(identity)\n            if any(not isinstance(v, Integral) or isinstance(v, bool) or not 0 <= v < len(label_names) for v in row[\'ner_tags\']):\n                raise ValueError(\'Invalid metadata label index.\')\n            gold = [label_names[v] for v in row[\'ner_tags\']]\n            for start, end, kind in metrics.entity_spans(gold):\n                counts[tuple(tokens[start:end])][kind] += 1\n            digest.update(canonical_bytes({\'id\': identity, \'tokens\': tokens, \'gold\': gold}))\n            n_sentences += 1\n            n_words += len(tokens)\n        if not n_sentences:\n            raise ValueError(\'Training rows must not be empty.\')\n        entries, ambiguous = [], []\n        for tokens, types in sorted(counts.items()):\n            if len(types) == 1:\n                kind = next(iter(types))\n                entries.append(dict(tokens=list(tokens), type=kind, mentions=types[kind]))\n            else:\n                ambiguous.append(dict(tokens=list(tokens), type_counts=dict(sorted(types.items()))))\n        summary = dict(train_sentences=n_sentences, train_words=n_words,\n            train_entity_mentions=sum(sum(c.values()) for c in counts.values()),\n            distinct_entity_surfaces=len(counts), retained_surfaces=len(entries),\n            ambiguous_surfaces=len(ambiguous), retained_mentions=sum(e[\'mentions\'] for e in entries),\n            excluded_ambiguous_mentions=sum(sum(e[\'type_counts\'].values()) for e in ambiguous),\n            max_retained_length=max((len(e[\'tokens\']) for e in entries), default=0))\n        return cls(dict(version=VERSION, policy=POLICY, label_names=list(label_names),\n            train_sha256=digest.hexdigest(), summary=summary, entries=entries, ambiguous=ambiguous))\n\n    def predict_sentence(self, tokens):\n        tokens = _tokens(tokens)\n        candidates = []\n        for start in range(len(tokens)):\n            node = self._trie\n            for end in range(start, len(tokens)):\n                node = node.get(tokens[end])\n                if node is None:\n                    break\n                if None in node:\n                    candidates.append((start, end + 1, node[None]))\n        occupied = [False] * len(tokens)\n        accepted = []\n        for start, end, kind in sorted(candidates, key=lambda m: (-(m[1] - m[0]), m[0])):\n            if not any(occupied[start:end]):\n                occupied[start:end] = [True] * (end - start)\n                accepted.append((start, end, kind))\n        tags = [\'O\'] * len(tokens)\n        for start, end, kind in sorted(accepted):\n            tags[start] = \'B-\' + kind\n            tags[start + 1:end] = [\'I-\' + kind] * (end - start - 1)\n        return tags\n\n    def predict(self, token_sentences):\n        return [self.predict_sentence(tokens) for tokens in token_sentences]\n\n\ndef _immutable_json(path, value):\n    path = Path(path)\n    content = canonical_bytes(value)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if path.exists():\n        if path.read_bytes() != content:\n            raise ValueError(\'Existing frozen/cached artifact differs: \' + str(path))\n    else:\n        with path.open(\'xb\') as stream:\n            stream.write(content)\n    if path.read_bytes() != content:\n        raise ValueError(\'Artifact read-back failed.\')\n\n\ndef freeze_model(project_root, model, protocol_sha256, source_hashes, data_source):\n    """Persist the dictionary and settings before even validation predictions."""\n    contract = dict(version=VERSION, policy=POLICY, model_sha256=fingerprint(model.payload),\n        protocol_sha256=protocol_sha256, source_code_sha256s=source_hashes,\n        # Observation time changes on pinned-source recovery, not model identity.\n        data_source={k:v for k,v in data_source.items() if k != \'recorded_at_utc\'}, bootstrap=BOOTSTRAP,\n        final_comparison_list=[\'memory baseline validation\', \'memory baseline test\', \'historical paper baseline (context only)\'],\n        train_only=True, tuning=False, frozen_before_validation_and_test=True)\n    identity = fingerprint(contract)\n    directory = Path(project_root)/\'config\'/VERSION/identity\n    _immutable_json(directory/\'model.json\', model.payload)\n    _immutable_json(directory/\'final_config.json\', contract)\n    receipt = dict(path=str(directory), contract_sha256=identity, model_sha256=contract[\'model_sha256\'],\n                   protocol_sha256=protocol_sha256, read_back_verified=True)\n    load_frozen(receipt)\n    return receipt\n\n\ndef load_frozen(receipt):\n    path = Path(receipt[\'path\'])\n    contract_bytes = (path/\'final_config.json\').read_bytes()\n    model_bytes = (path/\'model.json\').read_bytes()\n    if hashlib.sha256(contract_bytes).hexdigest() != receipt[\'contract_sha256\'] or hashlib.sha256(model_bytes).hexdigest() != receipt[\'model_sha256\']:\n        raise ValueError(\'Frozen configuration or model checksum changed.\')\n    contract = json.loads(contract_bytes)\n    if contract[\'model_sha256\'] != receipt[\'model_sha256\'] or contract[\'protocol_sha256\'] != receipt[\'protocol_sha256\'] or contract[\'policy\'] != POLICY or contract[\'bootstrap\'] != BOOTSTRAP:\n        raise ValueError(\'Frozen model contract mismatch.\')\n    # A restored cache cannot silently run under edited implementation code.\n    source_root = Path(__file__).parent\n    for name, digest in contract[\'source_code_sha256s\'].items():\n        if hashlib.sha256((source_root/name).read_bytes()).hexdigest() != digest:\n            raise ValueError(\'Frozen source differs: \' + name)\n    return MemoryBaseline(json.loads(model_bytes)), contract\n\n\ndef evaluate_cached(project_root, receipt, split, rows, label_names):\n    """Reuse verified saved predictions/scores on reruns; never refit on dev/test."""\n    if split not in (\'validation\', \'test\'):\n        raise ValueError(\'Only held-out validation or final test scoring is supported.\')\n    model, contract = load_frozen(receipt)\n    if list(label_names) != model.payload[\'label_names\']:\n        raise ValueError(\'Label metadata changed after freezing.\')\n    root = Path(project_root)\n    system = \'memory_\' + receipt[\'contract_sha256\'][:20]\n    path = root/\'results\'/\'predictions\'/(system + \'_\' + split + \'.jsonl\')\n    cache = root/\'results\'/VERSION/receipt[\'contract_sha256\']/(split + \'_evaluation.json\')\n    reused_predictions = path.exists()\n    reused_evaluation = cache.exists()\n    if reused_evaluation and not reused_predictions:\n        raise ValueError(\'Saved score exists without its predictions; restore the complete ZIP.\')\n    if not reused_predictions:\n        # The only input reaching predict is the original word list.\n        raw = model.predict([row[\'tokens\'] for row in rows])\n        prediction_io.write_predictions(path.parent, system, split, rows, raw, label_names, receipt[\'protocol_sha256\'])\n    records, meta = prediction_io.load_predictions(path, rows, label_names)\n    if meta[\'system\'] != system or meta[\'split\'] != split or meta[\'protocol_sha256\'] != receipt[\'protocol_sha256\'] or meta[\'prediction_iob2_repairs\'] != 0:\n        raise ValueError(\'Unexpected memory-baseline prediction metadata.\')\n    identity = dict(contract_sha256=receipt[\'contract_sha256\'], model_sha256=receipt[\'model_sha256\'],\n        protocol_sha256=receipt[\'protocol_sha256\'], split=split, predictions_sha256=meta[\'sha256\'],\n        bootstrap=BOOTSTRAP)\n    if reused_evaluation:\n        wrapper = json.loads(cache.read_text(encoding=\'utf-8\'))\n        result = wrapper[\'result\']\n        if wrapper[\'sha256\'] != fingerprint(result) or result[\'identity\'] != identity:\n            raise ValueError(\'Saved evaluation identity/checksum mismatch.\')\n    else:\n        ci = metrics.bootstrap_ci([r[\'gold_tags\'] for r in records], [r[\'raw_pred_tags\'] for r in records], **BOOTSTRAP)\n        result = dict(identity=identity, metrics=ci[\'point\'], bootstrap=ci)\n        _immutable_json(cache, dict(result=result, sha256=fingerprint(result)))\n    return result, dict(prediction_path=str(path), metadata_path=str(path.with_suffix(\'.meta.json\')),\n        evaluation_path=str(cache), prediction_sha256=meta[\'sha256\'],\n        evaluation_sha256=hashlib.sha256(cache.read_bytes()).hexdigest(),\n        reused_predictions=reused_predictions, reused_evaluation=reused_evaluation,\n        sentences=meta[\'sentences\'], words=meta[\'words\'], read_back_verified=True)\n\n\ndef dictionary_tables(model):\n    summary = model.payload[\'summary\']\n    type_table = []\n    lengths = Counter()\n    for kind in metrics.TYPES:\n        entries = [e for e in model.payload[\'entries\'] if e[\'type\'] == kind]\n        type_table.append(dict(type=kind, retained_surfaces=len(entries),\n            retained_mentions=sum(e[\'mentions\'] for e in entries),\n            excluded_ambiguous_mentions=sum(e[\'type_counts\'].get(kind, 0) for e in model.payload[\'ambiguous\'])))\n        for e in entries:\n            lengths[(kind, len(e[\'tokens\']))] += 1\n    if sum(r[\'retained_mentions\'] + r[\'excluded_ambiguous_mentions\'] for r in type_table) != summary[\'train_entity_mentions\']:\n        raise ValueError(\'Dictionary accounting mismatch.\')\n    return type_table, [dict(type=k, token_length=n, retained_surfaces=v) for (k,n),v in sorted(lengths.items())]\n\n\ndef toy_checks():\n    labels = [\'I-PER\',\'B-LOC\',\'O\',\'I-ORG\',\'B-PER\',\'I-LOC\',\'B-ORG\',\'B-MISC\',\'I-MISC\']\n    examples = [([\'New\',\'York\'],[\'B-LOC\',\'I-LOC\']), ([\'York\'],[\'B-ORG\']),\n        ([\'York\',\'University\',\'Press\'],[\'B-ORG\',\'I-ORG\',\'I-ORG\']),\n        ([\'Jordan\'],[\'B-PER\']), ([\'Jordan\'],[\'B-LOC\']),\n        ([\'Ana\'],[\'B-PER\']), ([\'Ana\'],[\'O\']),\n        ([\'A\',\'B\'],[\'B-LOC\',\'I-LOC\']), ([\'B\',\'C\'],[\'B-ORG\',\'I-ORG\'])]\n    rows = [dict(id=str(i),tokens=t,ner_tags=[labels.index(x) for x in y]) for i,(t,y) in enumerate(examples)]\n    model = MemoryBaseline.fit(rows, labels)\n    cases = [\n        (\'longer_at_same_start\',[\'New\',\'York\'],[\'B-LOC\',\'I-LOC\']),\n        (\'longer_crossing_later_start\',[\'New\',\'York\',\'University\',\'Press\'],[\'O\',\'B-ORG\',\'I-ORG\',\'I-ORG\']),\n        (\'ambiguous_discarded\',[\'Jordan\'],[\'O\']),\n        (\'case_sensitive\',[\'new\',\'York\'],[\'O\',\'B-ORG\']),\n        (\'adjacent_entities\',[\'Ana\',\'Ana\'],[\'B-PER\',\'B-PER\']),\n        (\'no_substring\',[\'Anabel\'],[\'O\']),\n        (\'no_invented_subphrase\',[\'University\'],[\'O\']),\n        (\'equal_length_earlier_start\',[\'A\',\'B\',\'C\'],[\'B-LOC\',\'I-LOC\',\'O\']),\n        (\'unseen_all_o\',[\'Nobody\',\'here\'],[\'O\',\'O\']),\n        (\'empty_input\',[],[]),\n    ]\n    checked = []\n    for name,tokens,expected in cases:\n        actual = model.predict_sentence(tokens)\n        if actual != expected:\n            raise ValueError(\'Synthetic decoder failed: \' + name)\n        metrics.entity_spans(actual)\n        checked.append(dict(case=name, tokens=tokens, expected=expected, actual=actual, passed=True))\n    if model.predict([[\'New\'],[\'York\']]) != [[\'O\'],[\'B-ORG\']]:\n        raise ValueError(\'Decoder crossed a sentence boundary.\')\n    return dict(synthetic_only=True, cases=checked, sentence_boundary_passed=True,\n                permuted_metadata_passed=True, o_is_not_entity_type_passed=True)\n'}
ACCEPTED_DATA = {'source': {'requested_source': 'eriktks/conll2003', 'actual_source': 'tomaarsen/conll2003', 'revision': '3a0c0677d3177b6122fceed3239dda7a5de6a514', 'format': 'parquet', 'source_url': 'https://huggingface.co/datasets/tomaarsen/conll2003/tree/3a0c0677d3177b6122fceed3239dda7a5de6a514', 'reason': 'Development document permits Parquet fallback; datasets 4.0 removes loading scripts.', 'license_as_stated_by_card': 'other', 'source_files': [{'path': 'data/train-00000-of-00001.parquet', 'size_bytes': 1241085, 'sha256': '28924b53e43ec4d741cbef16d8a6245a77f404a77f18d44e256f95fa29acb59b'}, {'path': 'data/validation-00000-of-00001.parquet', 'size_bytes': 315597, 'sha256': '0aa0bd59cf9ac8f60252d8194a8a223b650daba50f8e5a79284abf85420bfe84'}, {'path': 'data/test-00000-of-00001.parquet', 'size_bytes': 287552, 'sha256': '9242163e112f99b43c32b663c6d6f7d5df64de73e19e02fcac789935f72565b8'}, {'path': 'README.md', 'size_bytes': 14967, 'sha256': 'a6812e0cca383af9f4c316f8abe43169ddda7230c2686ba314629e59a41a66b9'}], 'preprocessing': 'None: no shuffle, split, filtering, label conversion or token normalization.', 'parity_scope': 'Split sizes, metadata, all-row structure/IOB2 and ordered content hashes checked. Byte-for-byte equivalence to eriktks/conll2003 has not been established.', 'recorded_at_utc': '2026-11-02T14:42:47.879150+00:00'}, 'saved_data': {'path': '/root/cs5489_ner_local/data/conll2003_20261102T144244Z-bb6d1daf', 'read_back_verified': True, 'files': [{'path': 'dataset_dict.json', 'size_bytes': 43, 'sha256': '1239dbd4ca38311b03319b672c05a6fdaacc884b0034eed047659406c62eedd4'}, {'path': 'test/data-00000-of-00001.arrow', 'size_bytes': 1614264, 'sha256': 'c640eee74c9972843f89bbebd941fcb18556d81218ded7184f13c21b6bf216f6'}, {'path': 'test/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'test/state.json', 'size_bytes': 249, 'sha256': 'bd644eb97e33c1a0ed4163a84a7aa1296cf5048e919728e2202f600977c146dd'}, {'path': 'train/data-00000-of-00001.arrow', 'size_bytes': 7056024, 'sha256': 'd62e5b846961b150ea80d24d2ee411f1eb63365b6aa9d91631057a61246b755f'}, {'path': 'train/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'train/state.json', 'size_bytes': 250, 'sha256': '3ff00933cf5e8e832682190ebe6b245a13345ee6592912f19aa2c396aa424b81'}, {'path': 'validation/data-00000-of-00001.arrow', 'size_bytes': 1769816, 'sha256': 'ab07b4269a275b8a87b240b9169e245a647ab95353027f417118f321d16d5ee8'}, {'path': 'validation/dataset_info.json', 'size_bytes': 3617, 'sha256': '1d6c341e46278358fea4c53476e7b18c44214236df60a9d5b79470e193f248d7'}, {'path': 'validation/state.json', 'size_bytes': 255, 'sha256': '95534551937bb664274088bfd58d5b18afc30934e11f42c54ba456f0827c70d0'}]}, 'audit': {'splits': {'train': {'rows': 14041, 'tokens': 203621, 'ordered_core_sha256': 'aa1208d2f06bc11610d0d9527af175e282ca0f4092a06a5a9381240e1dfece6a', 'id_order_verified': True, 'document_order_verified': True, 'documents': 946}, 'validation': {'rows': 3250, 'tokens': 51362, 'ordered_core_sha256': '91c3c12a9140e03ca679f24dc0437cdbae033babea946388677e46f9a52a78c2', 'id_order_verified': True, 'document_order_verified': True, 'documents': 216}, 'test': {'rows': 3453, 'tokens': 46435, 'ordered_core_sha256': 'bbdf60c2c1c3fbd1e7cb6a865d9d4dfd9df0110dff115de49f972175df7d3882', 'id_order_verified': True, 'document_order_verified': True, 'documents': 231}}, 'label_names': {'pos_tags': ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'], 'chunk_tags': ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'], 'ner_tags': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']}, 'iob2_violations': 0, 'hash_format': 'core-fields-decoded-tags-jsonl-utf8-v1'}, 'recovery': 'redownloaded-pinned-step1-source', 'run_id': '20261102T144244Z-bb6d1daf', 'acceptance': {'date': '2026-11-02', 'evidence': 'NER_step4_20261102T144254Z_65558b19.zip', 'archive_sha256': '2d4ea33ff624ef834b0a768aa59caf91c72e4d1b612284b9f1caad1aae832bc0'}}
ACCEPTED_PROTOCOL = {'change_control': 'Never overwrite this v1 contract. Any necessary design change gets a new version and rationale before model selection/test scoring; keep the old contract. Register exact search spaces/budgets before search and final model settings before test.', 'classical': {'crf_fallback': {'folds': 3, 'report': 'Identify fallback explicitly; no switching based on CV/dev/test performance.', 'search': 'randomized, 10 configurations, seed 42', 'trigger': 'A training-only timing pilot predicts >4 CPU hours for the registered 5-fold search; log estimate and switch before viewing search scores.'}, 'features': 'F3 gold POS/chunk is research-only; deployment variants use raw-text-available features.', 'leakage_rule': 'Fit vectorizers, learned vocabularies, scalers, SVD/PCA, K-means and all other learned transforms only on the fit portion of each fold. Refit selected transforms/model on train, never dev/test.', 'primary_folds': 5, 'selection': 'Mean of held-out fold micro entity F1, not token accuracy; report fold std (ddof=1).', 'tie_break': 'First configuration in the pre-registered simpler-to-more-complex search order.'}, 'dataset': {'audit': {'hash_format': 'core-fields-decoded-tags-jsonl-utf8-v1', 'iob2_violations': 0, 'label_names': {'chunk_tags': ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'], 'ner_tags': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], 'pos_tags': ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']}, 'splits': {'test': {'document_order_verified': True, 'documents': 231, 'id_order_verified': True, 'ordered_core_sha256': 'bbdf60c2c1c3fbd1e7cb6a865d9d4dfd9df0110dff115de49f972175df7d3882', 'rows': 3453, 'tokens': 46435}, 'train': {'document_order_verified': True, 'documents': 946, 'id_order_verified': True, 'ordered_core_sha256': 'aa1208d2f06bc11610d0d9527af175e282ca0f4092a06a5a9381240e1dfece6a', 'rows': 14041, 'tokens': 203621}, 'validation': {'document_order_verified': True, 'documents': 216, 'id_order_verified': True, 'ordered_core_sha256': '91c3c12a9140e03ca679f24dc0437cdbae033babea946388677e46f9a52a78c2', 'rows': 3250, 'tokens': 51362}}}, 'revision': '3a0c0677d3177b6122fceed3239dda7a5de6a514', 'source': 'tomaarsen/conll2003', 'source_files': [{'path': 'data/train-00000-of-00001.parquet', 'sha256': '28924b53e43ec4d741cbef16d8a6245a77f404a77f18d44e256f95fa29acb59b', 'size_bytes': 1241085}, {'path': 'data/validation-00000-of-00001.parquet', 'sha256': '0aa0bd59cf9ac8f60252d8194a8a223b650daba50f8e5a79284abf85420bfe84', 'size_bytes': 315597}, {'path': 'data/test-00000-of-00001.parquet', 'sha256': '9242163e112f99b43c32b663c6d6f7d5df64de73e19e02fcac789935f72565b8', 'size_bytes': 287552}, {'path': 'README.md', 'sha256': 'a6812e0cca383af9f4c316f8abe43169ddda7230c2686ba314629e59a41a66b9', 'size_bytes': 14967}]}, 'fold_plan_sha256': {'train_3fold': 'b654bd800ed127210f6edc8ec61fbf4ef0b2f8b86e37b1c98f9b8f03cd72c6bd', 'train_5fold': '210397396c7ac1595d8fd91cb876ae36800180f044ba75efe439b41fc4ae0bbd'}, 'label_names': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], 'metric_contract': {'bootstrap': '2000 sentence-level draws with replacement, 95% percentile intervals, NumPy default_rng seed 2026; sum TP/FP/FN then compute corpus metrics.', 'gold': 'Original metadata-decoded IOB2; invalid gold or mismatched sentence/token lengths raises an error.', 'missing_types': 'Always output PER/ORG/LOC/MISC; undefined precision, recall or F1 is 0, with explicit support and denominator counts.', 'paired': 'B minus A on identical sentence indices per draw; require identical ids, tokens and gold. An interval excluding zero is descriptive evidence, not a calibrated p-value.', 'prediction': 'Before all scoring, an I-X not preceded by B-X/I-X becomes B-X. Other tags unchanged. Same fixed rule for every system; preserve raw tags and count repairs.', 'primary': 'Micro entity precision/recall/F1; exact start, exclusive end and type, within each original sentence.', 'rationale': 'Matches development-document Step 9 orphan-I extraction rule; does not inspect gold labels. Boundary/type matching remains exact.', 'scale': 'All metrics and confidence limits are fractions in [0,1]; paired differences in [-1,1].', 'seqeval': '1.2.2, mode=strict, scheme=IOB2, zero_division=0; independently checked against span counts.', 'uncertainty_scope': 'Conditional on fixed trained models and sentence resampling; does not include training-seed variance or within-document dependence.', 'version': 'ner-entity-metrics-v1'}, 'prediction_schema': {'alignment': 'Exactly one row per original sentence, one tag per original word. Reassemble Transformer chunks first.', 'fields': {'gold_tags': 'metadata-decoded, valid IOB2 tags', 'pred_tags': 'canonical IOB2 predictions', 'raw_pred_tags': 'unaltered model tag strings before the fixed orphan-I rule', 'sentence_id': 'unique string; original dataset id', 'tokens': 'original word strings in order'}, 'filename': '<system>_<split>.jsonl plus .meta.json sidecar', 'sidecar': 'schema/metric version, system, split, protocol hash, counts and JSONL SHA-256; no overwrite.', 'test_use': 'Write final test predictions only after final settings are frozen; re-use saved files for bootstrap, never retrain during resampling.', 'version': 'ner-word-predictions-v1'}, 'public_source_sha256': {'data.py': 'f1e86e664ba29c02987cad7072e0b4184511744313bdef8fc9ea99e2bb6a2ab2', 'experiment_protocol.py': '980f7d17db1e868ca9c886afeb98f3198f9ce5f07b571668c714062055fc3964', 'local_backup.py': '193692b3a886f8d956ecee73c8eb076928e84011d7cc5626bf6901308ea6cbfc', 'metrics.py': '774b209d2963f071abd7e0dca58e60efa2b7d3e427d0e8ed9d11714f2c7039d5', 'prediction_io.py': '072cb678b94c7d72f02ce18270d19356b8ae6ab3d2867089d78078515597600c', 'step1_data.py': '3490f52b9ae4de8627aa2f94270ad04cb61f67a327ec46bf0f60df242f3faffe', 'step2_eda.py': '220c1cd4e95c56dc63db07b9e28dcb6f022ff45fc5aaa52bd8b96d535bf55d4b', 'step3_alignment.py': 'a919a891bfbbdfafbea04634373b08498058d8d25ac44c62a5890fcbf65731d8'}, 'seeds': {'bootstrap': 2026, 'classical': 42, 'random_search': 42, 'representation_sampling': 42, 'transformer': [42, 52, 62]}, 'split_usage': {'test': 'Only after final settings/checkpoints and comparison list are frozen. Score each final checkpoint once; save predictions for all later uncertainty analysis. No tuning or error-driven changes.', 'train': 'All fitting. Classical hyperparameters use the saved 5 contiguous document-safe training folds.', 'validation': 'Compare methods/features; choose Transformer hyperparameters/checkpoints/early stopping; perform all error analysis.'}, 'status': 'frozen experimental design; model-specific grids must be registered before their search', 'step4_scope': 'Real data integrity/fold generation only. Metric/bootstrap/prediction-file demonstrations are synthetic. No real model predictions or test scores are generated.', 'transformer': {'encoders': ['bert-base-cased', 'roberta-base'], 'full_kfold': False, 'max_length': 256, 'preprocessing': 'Accepted Step 3 first-subword supervision; reconstruct original sentences before scoring.', 'primary_comparison_seed': 42, 'reason': 'Full K-fold fine-tuning is too costly; held-out development selection plus independent seed repeats.', 'report': 'Report all three seeds and mean/sample std (ddof=1). Never select the best test seed.', 'selection': 'Tune on validation with seed 42, freeze settings, then repeat the chosen setting for seeds 42/52/62. Each seed selects its checkpoint by validation F1.'}, 'uncertainty': {'confidence': 0.95, 'method': 'sentence percentile', 'n_resamples': 2000, 'paired_rule': 'Same original sentence indices for A and B; resample saved predictions, no refitting.', 'preplanned_primary_pairs': ['selected deployable CRF vs BERT seed 42', 'BERT seed 42 vs RoBERTa seed 42'], 'report': 'B-minus-A intervals on corpus micro F1. Per-type comparisons exploratory; intervals are not multiple-comparison corrected or training-variance intervals.', 'seed': 2026}, 'version': 'ner-experiment-v1'}
ACCEPTED_PROTOCOL_SHA = 'f70ce61993c32c158900623a495c61efd0a70c95fadbdc2cfb572cf9d01cec01'
# Bootstrap only the reviewed backup helper; never execute code from an uploaded ZIP.
import types
backup=types.ModuleType('local_backup')
exec(compile(SOURCE_BUNDLE['local_backup.py'],'embedded_local_backup.py','exec'),backup.__dict__)
if RESTORE_STEP5A_ZIP is not None:
    manifest=backup.verify_backup(RESTORE_STEP5A_ZIP)
    if manifest['step']!='step5a' or manifest['revision'] not in ('2026-11-03-memory-v1',NOTEBOOK_REVISION):
        raise ValueError('Restore a Step 5a ZIP of this revision, not a different step.')
    PROJECT_ROOT=PROJECT_ROOT.parent/('cs5489_ner_restored_'+RUN_ID)
    backup.restore_backup(RESTORE_STEP5A_ZIP,PROJECT_ROOT)
for folder in ('src','config','logs','results','data'):
    (PROJECT_ROOT/folder).mkdir(parents=True,exist_ok=True)
for name,source in SOURCE_BUNDLE.items():
    target=PROJECT_ROOT/'src'/name
    if target.exists() and target.read_text(encoding='utf-8')!=source:
        raise ValueError('Existing source differs; keep it and investigate: '+str(target))
    target.write_bytes(source.encode('utf-8'))
def load_project_module(name):
    spec=importlib.util.spec_from_file_location(name,PROJECT_ROOT/'src'/(name+'.py'))
    module=importlib.util.module_from_spec(spec)
    sys.modules[name]=module
    spec.loader.exec_module(module)
    return module
backup=load_project_module('local_backup')
data_tools=load_project_module('step1_data')
eda_tools=load_project_module('step2_eda')
alignment=load_project_module('step3_alignment')
metrics=load_project_module('metrics')
prediction_io=load_project_module('prediction_io')
shared_data=load_project_module('data')
experiment=load_project_module('experiment_protocol')
memory=load_project_module('step5a_memory')
RESULTS_DIR=PROJECT_ROOT/'results'/f'step5a_{RUN_ID}'
RESULTS_DIR.mkdir(parents=True,exist_ok=False)
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
SOURCE_HASHES={name:backup.sha256_file(PROJECT_ROOT/'src'/name) for name in SOURCE_BUNDLE}
for name,digest in ACCEPTED_PROTOCOL['public_source_sha256'].items():
    if SOURCE_HASHES[name]!=digest: raise ValueError('Step 4 public source changed: '+name)
data_tools.write_json(PROJECT_ROOT/'config'/'step5a_accepted_data_reference.json',ACCEPTED_DATA)
print('Project:',PROJECT_ROOT)
print('Results:',RESULTS_DIR)
print('Reviewed public interfaces loaded; no model fitted yet.')


## 5a.3 验证数据与第四步协议
验证来源、句序、标签及三划分完整性；恢复的训练折必须与第四步哈希完全相同。
测试集在此仅作完整性核对，尚不预测或评分。不需要重跑第三步分词。


In [ ]:
STEP5A_COMPLETE=DATA_READY=MODEL_FROZEN=VALIDATION_READY=TEST_READY=BACKUP_VERIFIED=BACKUP_DOWNLOADED=False
raw_datasets,DATA_RECORD,LABEL_MAPPING=shared_data.load_verified_dataset(PROJECT_ROOT,ACCEPTED_DATA,RUN_ID)
LABEL_NAMES=LABEL_MAPPING['label_names']
if LABEL_NAMES!=ACCEPTED_PROTOCOL['label_names'] or metrics.METRIC_CONTRACT!=ACCEPTED_PROTOCOL['metric_contract']:
    raise ValueError('Label order or metric contract changed.')
if experiment.fingerprint(ACCEPTED_PROTOCOL)!=ACCEPTED_PROTOCOL_SHA:
    raise ValueError('Accepted protocol fingerprint changed.')
TRAIN_ROWS=list(raw_datasets['train'])
FOLD_PLANS={f'train_{n}fold':experiment.make_contiguous_folds(TRAIN_ROWS,n) for n in (5,3)}
for name,plan in FOLD_PLANS.items():
    experiment.verify_fold_plan(plan,TRAIN_ROWS)
    if experiment.fingerprint(plan)!=ACCEPTED_PROTOCOL['fold_plan_sha256'][name]:
        raise ValueError('Fold plan differs from accepted Step 4: '+name)
PROTOCOL_RECEIPT=experiment.freeze_protocol(PROJECT_ROOT,ACCEPTED_PROTOCOL,FOLD_PLANS)
if PROTOCOL_RECEIPT['protocol_sha256']!=ACCEPTED_PROTOCOL_SHA: raise ValueError('Protocol changed.')
data_tools.write_json(PROJECT_ROOT/'config'/'label_mapping.json',LABEL_MAPPING)
DATA_READY=True
print('Rows:',{s:r['rows'] for s,r in DATA_RECORD['audit']['splits'].items()})
print('Data recovery:',DATA_RECORD['recovery'])
print('Step 4 protocol preserved:',ACCEPTED_PROTOCOL_SHA)


## 5a.4 合成检查、拟合训练词典并冻结最终配置
先检查最长匹配、歧义剔除、大小写、整词匹配、句界与标签顺序。
只读取官方训练集构建完整实体词典；不做频率筛选，不把开发集加入词典。
将词典、歧义条目、规则、源码哈希与测试比较清单保存并读回，然后才允许评分。


In [ ]:
from IPython.display import display
STEP5A_COMPLETE=MODEL_FROZEN=VALIDATION_READY=TEST_READY=BACKUP_VERIFIED=BACKUP_DOWNLOADED=False
if not DATA_READY: raise RuntimeError('Complete 5a.3 first.')
TOY_CHECKS=memory.toy_checks()
METRIC_TOYS=metrics.toy_checks()
data_tools.write_json(RESULTS_DIR/'synthetic_checks.json',dict(decoder=TOY_CHECKS,metrics=METRIC_TOYS))
MODEL=memory.MemoryBaseline.fit(TRAIN_ROWS,LABEL_NAMES)
MODEL_RECEIPT=memory.freeze_model(PROJECT_ROOT,MODEL,ACCEPTED_PROTOCOL_SHA,SOURCE_HASHES,DATA_RECORD['source'])
RESTORED_MODEL,FINAL_CONFIG=memory.load_frozen(MODEL_RECEIPT)
if RESTORED_MODEL.payload!=MODEL.payload: raise ValueError('Saved dictionary read-back mismatch.')
DICT_TYPES,DICT_LENGTHS=memory.dictionary_tables(RESTORED_MODEL)
pd.DataFrame(DICT_TYPES).to_csv(RESULTS_DIR/'dictionary_by_type.csv',index=False)
pd.DataFrame(DICT_LENGTHS,columns=['type','token_length','retained_surfaces']).to_csv(RESULTS_DIR/'dictionary_lengths.csv',index=False)
data_tools.write_json(RESULTS_DIR/'dictionary_summary.json',MODEL.payload['summary'])
data_tools.write_json(RESULTS_DIR/'model_receipt.json',MODEL_RECEIPT)
MODEL_FROZEN=True
print('Decoder checks: 10 cases + sentence boundary PASS; metric toys: 10 PASS')
print('Train-only dictionary:',MODEL.payload['summary'])
print('FINAL CONFIG FROZEN BEFORE VALIDATION/TEST:',MODEL_RECEIPT['contract_sha256'])
display(pd.DataFrame(DICT_TYPES))


## 5a.5 开发集：跑通完整评估与保存接口
模型只接收 tokens；预测写成第四步固定的逐句 JSONL，然后读回核对 ids、tokens、gold 与长度。
由保存的预测计算整体/四类型实体指标与 95% 置信区间。分数内部为 0–1，显示为百分制。
已有同一冻结模型的完整结果时复用，避免重复预测。不得根据这里的结果改动本基线配置。


In [ ]:
STEP5A_COMPLETE=VALIDATION_READY=TEST_READY=BACKUP_VERIFIED=BACKUP_DOWNLOADED=False
if not MODEL_FROZEN: raise RuntimeError('Complete 5a.4 first.')
VALIDATION_ROWS=list(raw_datasets['validation'])
VALIDATION_RESULT,VALIDATION_RECEIPT=memory.evaluate_cached(PROJECT_ROOT,MODEL_RECEIPT,'validation',VALIDATION_ROWS,LABEL_NAMES)
def display_scores(result):
    report=result['metrics']
    records=[dict(type='overall',**report['overall']),*[dict(type=k,**v) for k,v in report['per_type'].items()]]
    frame=pd.DataFrame(records)
    for metric in ('precision','recall','f1'): frame[metric]*=100
    display(frame)
display_scores(VALIDATION_RESULT)
VALIDATION_READY=True
print('VALIDATION PIPELINE PASS; predictions and evaluation reused:',VALIDATION_RECEIPT['reused_predictions'],VALIDATION_RECEIPT['reused_evaluation'])


## 5a.6 固定基线的一次最终测试
没有搜索或配置选择；使用 5a.4 已冻结的相同词典与规则。
首次运行保存最终测试预测；重跑读取相同缓存。Bootstrap 仅重采样保存的预测，不重新拟合模型。
历史分数差异只作描述，不作为测试驱动改代码的理由，也不把置信区间解释成 p 值。


In [ ]:
STEP5A_COMPLETE=TEST_READY=BACKUP_VERIFIED=BACKUP_DOWNLOADED=False
if not (MODEL_FROZEN and VALIDATION_READY): raise RuntimeError('Complete 5a.4–5a.5 first.')
memory.load_frozen(MODEL_RECEIPT)
TEST_ROWS=list(raw_datasets['test'])
TEST_RESULT,TEST_RECEIPT=memory.evaluate_cached(PROJECT_ROOT,MODEL_RECEIPT,'test',TEST_ROWS,LABEL_NAMES)
display_scores(TEST_RESULT)
TEST_READY=True
print('Final test predictions and evaluation reused:',TEST_RECEIPT['reused_predictions'],TEST_RECEIPT['reused_evaluation'])
print('Measured test F1:',round(100*TEST_RESULT['metrics']['overall']['f1'],4))
print('Paper reference F1: 59.61; difference in percentage points:',round(100*(TEST_RESULT['metrics']['overall']['f1']-memory.PAPER['f1']),4))
print('No tuning, refitting or error-driven changes on test.')


## 5a.7 保存报告与下载 ZIP
导出 5 张 CSV、完整 JSON 报告、词典/歧义记录、冻结配置、两个划分的预测及 bootstrap 结果。
词典是本模型的完整检查点，随 ZIP 保留；数据缓存仍排除。CPU 即可，不产生大型权重。
文件中指标统一为 0–1；历史对比表同时给出百分制差值。置信区间不包含训练随机性或文章内句子相关性。


In [ ]:
STEP5A_COMPLETE=BACKUP_VERIFIED=BACKUP_DOWNLOADED=False
if not (DATA_READY and MODEL_FROZEN and VALIDATION_READY and TEST_READY): raise RuntimeError('Complete 5a.3–5a.6 first.')
memory.load_frozen(MODEL_RECEIPT)
EVALUATIONS={'validation':VALIDATION_RESULT,'test':TEST_RESULT}
EVALUATION_RECEIPTS={'validation':VALIDATION_RECEIPT,'test':TEST_RECEIPT}
metric_rows=[]; interval_rows=[]
for split,result in EVALUATIONS.items():
    report=result['metrics']
    metric_rows.extend([dict(split=split,type='overall',**report['overall']),
        *[dict(split=split,type=k,**v) for k,v in report['per_type'].items()]])
    interval_rows.extend(dict(split=split,type=k,metric=m,**v) for k,group in result['bootstrap']['intervals'].items() for m,v in group.items())
pd.DataFrame(metric_rows).to_csv(RESULTS_DIR/'heldout_metrics.csv',index=False)
pd.DataFrame(interval_rows).to_csv(RESULTS_DIR/'bootstrap_intervals.csv',index=False)
PAPER_COMPARISON=[dict(metric=m,measured=TEST_RESULT['metrics']['overall'][m],paper_reference=v,
    difference_percentage_points=100*(TEST_RESULT['metrics']['overall'][m]-v),
    source=memory.POLICY['reference'],interpretation='Historical context only; not a tuning target or exact-reproduction claim.') for m,v in memory.PAPER.items()]
pd.DataFrame(PAPER_COMPARISON).to_csv(RESULTS_DIR/'paper_comparison.csv',index=False)
REPORT=dict(run_id=RUN_ID,notebook_revision=NOTEBOOK_REVISION,accepted_reference=ACCEPTED_DATA['acceptance'],
    data_source=DATA_RECORD['source'],data_audit=DATA_RECORD['audit'],label_mapping=LABEL_MAPPING,
    protocol_sha256=ACCEPTED_PROTOCOL_SHA,policy=memory.POLICY,model_receipt=MODEL_RECEIPT,
    dictionary_summary=MODEL.payload['summary'],dictionary_by_type=DICT_TYPES,dictionary_lengths=DICT_LENGTHS,
    evaluations=EVALUATIONS,evaluation_receipts=EVALUATION_RECEIPTS,paper_comparison=PAPER_COMPARISON,
    synthetic_checks=TOY_CHECKS,fit_split='train',hyperparameter_search=False,
    final_config_frozen_before_validation_and_test=True,gate1_pipeline_checks_passed=True,
    acceptance='Awaiting computer ZIP download and external review.',test_error_analysis=False)
data_tools.write_json(RESULTS_DIR/'step5a_report.json',REPORT)
(RESULTS_DIR/'HANDOFF.md').write_text(
    '# Step 5a memory baseline\n\n'
    'Fit: official train only, complete entity token tuples, one unique NE type. Ambiguous surfaces discarded; O is not an NE type.\n'
    'Decode: exact case-sensitive words, all sentence-local candidates sorted by length descending then start ascending; skip occupied tokens. Output valid IOB2, adjacent B tags preserved. Predict never receives gold/POS/chunk.\n'
    'The paper does not fully specify overlap ties/casing; these explicit project choices were frozen before any held-out score. This is not a claim of bit-for-bit historical reproduction.\n'
    'src/step5a_memory.py: MemoryBaseline.fit(train_rows, label_names); predict(token_sentences); freeze_model; load_frozen; evaluate_cached. Shared Step 4 APIs and protocol are unchanged.\n'
    'The full model dictionary and ambiguity counts are in config/ner-memory-baseline-v1/<contract>/model.json, with final_config.json. No large weights or new tokenizer.\n'
    'Original-sentence JSONL predictions and sidecars are in results/predictions/. Scores and 2000-draw, 95% sentence bootstrap caches are in results/ner-memory-baseline-v1/<contract>/. All metrics/intervals use fractions.\n'
    'Same-model reruns verify and reuse saved predictions/evaluations. If the runtime is lost AFTER a prior Step 5a run, upload your verified Step 5a ZIP using the Files panel and set RESTORE_STEP5A_ZIP in section 5a.2 to restore to a new directory. Full-corpus data are recovered from the pinned public source.\n'
    'Retain every earlier ZIP and download this notebook separately. The ZIP contains the complete small baseline model but excludes raw datasets/cache.\n'
    'Gate 1 pipeline checks are complete only after this notebook runs successfully; external acceptance still requires the user download and ZIP review. Next: Step 5b LR/SVM.\n'
    'Reference: https://aclanthology.org/W03-0419.pdf section 3.4/Table 5. P=71.91/R=50.90/F1=59.61 are published percentages, not substituted experiment scores. No threshold forcing agreement.\n',encoding='utf-8')
TABLE_COUNT=len(list(RESULTS_DIR.glob('*.csv')))
if TABLE_COUNT!=5: raise ValueError('Expected five Step 5a CSV tables.')
artifact_paths=set(p for p in RESULTS_DIR.rglob('*') if p.is_file())
artifact_paths.update(Path(MODEL_RECEIPT['path'])/name for name in ('model.json','final_config.json'))
for receipt in EVALUATION_RECEIPTS.values():
    artifact_paths.update(Path(receipt[k]) for k in ('prediction_path','metadata_path','evaluation_path'))
ARTIFACTS=[dict(path=p.relative_to(PROJECT_ROOT).as_posix(),size_bytes=p.stat().st_size,sha256=backup.sha256_file(p)) for p in sorted(artifact_paths)]
data_tools.write_json(PROJECT_ROOT/'logs'/f'step5a_environment_{RUN_ID}.json',dict(
    run_id=RUN_ID,notebook_revision=NOTEBOOK_REVISION,python=sys.version,platform=platform.platform(),
    packages={p:metadata.version(p) for p in [*REQUIRED,'scipy','pyarrow']},source_code_sha256s=SOURCE_HASHES,
    data_recovery=DATA_RECORD['recovery'],protocol_sha256=ACCEPTED_PROTOCOL_SHA,
    model_contract_sha256=MODEL_RECEIPT['contract_sha256'],artifacts=ARTIFACTS))
data_tools.write_json(PROJECT_ROOT/'config'/'memory_latest.json',dict(run_id=RUN_ID,results_dir=str(RESULTS_DIR),
    model_receipt=MODEL_RECEIPT,artifacts=ARTIFACTS,verified=True,computer_download_confirmed=False))
BACKUP_RECEIPT=backup.create_backup(PROJECT_ROOT,EXPORT_DIR,step='step5a',revision=NOTEBOOK_REVISION,include_checkpoints=False)
BACKUP_PATH=Path(BACKUP_RECEIPT['path'])
BACKUP_VERIFIED=BACKUP_RECEIPT['archive_verified']
print('ZIP READY — check the computer copy after downloading.')
print(json.dumps(BACKUP_RECEIPT,indent=2))


In [ ]:
if not BACKUP_VERIFIED or not BACKUP_PATH.is_file(): raise RuntimeError('Generate the verified ZIP first.')
try:
    from google.colab import files
    files.download(str(BACKUP_PATH))
    print('Download requested:',BACKUP_PATH.name)
except Exception as exc:
    print('Automatic download did not finish:',type(exc).__name__,str(exc))
print('Files-panel fallback path:',BACKUP_PATH)
print('Also save this notebook with File > Download .ipynb.')


## 5a.8 确认电脑备份并反馈
确认电脑已有同名、字节数一致的 ZIP 后，把 `BACKUP_DOWNLOADED` 改为 `True`，再运行。
发送最后输出，把 ZIP 放进电脑 `CS 5489/NER` 文件夹供检查。若报错，发送小节与错误文本。
不要因为结果没有刚好等于历史分数而调整匹配规则。


In [ ]:
BACKUP_DOWNLOADED=False  # Set True only after checking the computer ZIP.
STEP5A_COMPLETE=False
if not (DATA_READY and MODEL_FROZEN and VALIDATION_READY and TEST_READY and BACKUP_VERIFIED): raise RuntimeError('Complete 5a.3–5a.7 first.')
if not BACKUP_DOWNLOADED:
    print('WAITING: check the computer ZIP, then set BACKUP_DOWNLOADED=True.')
else:
    backup.verify_backup(BACKUP_PATH)
    if backup.sha256_file(BACKUP_PATH)!=BACKUP_RECEIPT['sha256']: raise RuntimeError('ZIP changed after generation.')
    STEP5A_COMPLETE=True
    print('STEP 5a COMPLETE: train-only memory baseline, evaluation and ZIP verified; computer download confirmed by the user.')
    print('Dictionary:',MODEL.payload['summary'])
    for split,result in EVALUATIONS.items():
        overall=result['metrics']['overall']; ci=result['bootstrap']['intervals']['overall']['f1']
        print(split,'P/R/F1 (%):',[round(100*overall[k],4) for k in ('precision','recall','f1')],
              'F1 95% CI (%):',[round(100*ci[k],4) for k in ('low','high')])
    print('Paper test F1: 59.61 (reference only); difference (pp):',round(100*(TEST_RESULT['metrics']['overall']['f1']-memory.PAPER['f1']),4))
    print('Prediction JSONL save/read-back: PASS; word counts:',{s:r['words'] for s,r in EVALUATION_RECEIPTS.items()})
    print('Protocol SHA-256:',ACCEPTED_PROTOCOL_SHA)
    print('Model config SHA-256:',MODEL_RECEIPT['contract_sha256'])
    print('Bootstrap: 2000 sentence draws; 95% percentile; seed 2026')
    print('Tables:',TABLE_COUNT)
    print('Results:',RESULTS_DIR)
    print('Backup:',BACKUP_RECEIPT['name'])
    print('Bytes:',BACKUP_RECEIPT['size_bytes'])
    print('SHA-256:',BACKUP_RECEIPT['sha256'])
    print('Gate 1 pipeline checks passed; send this output and ZIP for external acceptance before Step 5b.')
